# 02 — Models and transfer

Nine model families are fitted on MCS Sweep 6 and applied to YRBS 2023. This notebook asks what
survives the crossing, what an importing authority can buy back, and whether either answer
depends on how the outcome was defined.

## A — Research questions and experiment map

Three questions, in the order the argument needs them.

1. **How much of a welfare-risk model survives being carried across cohorts?** Split into
   discrimination and calibration, because there is no reason for the two to survive together.
2. **What can an importing authority buy back, and at what price?** Three prices: nothing,
   five hundred target outcomes, and a range of budgets in between.
3. **Does any of it depend on how the outcome was defined, or on which families were used?**

### What this notebook decides, and what it inherits

Section D derives its own model selection. One procedure is applied to each cohort separately —
development seeds 0, 1 and 2, five-fold stratified inner cross-validation inside that cohort's
own outer-training partitions, AUC, the mean of the three seed-level means — and it fixes one
configuration per family and threshold, 27 per cohort and 54 in total. The selection writes two
private records and then stops at a promotion gate: `spec/local_model_settings.csv` is written
only by `scripts/promote_local_settings.py`, after review. Section D fits under the promoted
configuration, and Section F asks whether it made a difference.

Three kinds of number appear below:

| | |
|---|---|
| **loaded** | selected in Section D, promoted by a separate reviewed command, then read back from the tracked spec — the hyperparameters |
| **recomputed** | derived from the cohorts here — every result in Sections E to M |
| **frozen** | published under `outputs/`, from an earlier run, never rewritten here |

### The order — the configuration, then what it achieves, then whether it mattered

| | |
|---|---|
| **A** | research questions and experiment map |
| **B** | data, outcomes, splits, and what the split checks establish |
| **C** | preprocessing: what cohort standardisation is doing |
| **D** | model selection, its review, the reviewed promotion, then the canonical source models |
| **E** | the two references and the main transfer baseline |
| **F** | did the loaded configuration matter? |
| **G** | label-free adaptation |
| **H** | label-using adaptation at k = 500 |
| **I** | post-training probability adjustment |
| **J** | label budgets |
| **K** | calibration correction |
| **L** | structure transfer |
| **M** | outcome robustness |
| **N** | limitations |
| **O** | summary, and the four tables and one score file this run writes |

D comes before F because they are the same models. The tuned arm of the configuration
comparison is the source reference, the target-trained reference and the unadapted baseline —
the same estimator, the same hyperparameters, the same seed, the same training frames. Fitting
the models once and reading Section F's tuned column off Sections D and E is the same
arithmetic as fitting them twice and checking that the two agree.

**Every experiment is one cell and binds one dataframe.** A comparison concatenates the frames
it needs, by name. Nothing accumulates in a shared structure, nothing is written while the
notebook runs, and nothing is read back from disk: a cell whose predecessor has not been run
fails on an unbound name rather than quietly finding a file an earlier run left behind.

**Each experiment carries a status.** *Primary analysis* answers one of the three questions
above; *diagnostic* checks that the machinery does what it claims; *exploratory follow-up* was
prompted by an observed result rather than planned; *robustness* varies something the main
result should not depend on. **No status claims pre-registration** — nothing in this repository
records a hypothesis or a decision rule written down before the corresponding result was
inspected.

In [ ]:
import sys
import time
from itertools import product

# src/ is a sibling of notebooks/; only one of these two exists from any
# working directory, and a path that does not exist is ignored.
sys.path[:0] = ["src", "../src"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared style for this notebook's figures. They are shown, not saved: nothing here writes a
# figure to either root. The one published figure is notebook 03's, which sets its own style
# for the column it has to fit.
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
                     "figure.dpi": 110})

import config
import paper
import regime_names
import data
import evaluation
import inputs
import models
import outcomes
import publication
import scores
import transfer

# Presentation mode, and nothing else. False is the internal notebook and is what a run
# executes: the full working grids and every diagnostic are shown.
# scripts/make_public_notebooks.py writes a SEPARATE copy with this set to True, which
# narrows the visible flow to the headline threshold and the compact result after each
# experiment, and keeps output only in the cells spec/public_notebook_cells.json names.
# Both settings fit, score, validate and write exactly the same files.
PUBLIC_NOTEBOOK = False

# The threshold the main text leads on. >=1 and >=3 are computed throughout and reported in
# the sensitivity table at the end.
HEADLINE_THRESHOLD = ">=2"

### The experiment map

One row per experiment, in the order they appear. `status` is the claim each one is entitled to
make; `cost` is what it fits, because a few of these dominate the runtime and it is worth
knowing which before starting a run.

In [ ]:
EXPERIMENTS = [
    # section, experiment, status, what it answers, cost
    ("C", "cohort-standardisation audit",  "diagnostic",
     "how much adaptation is inside the baseline", "no fits"),
    ("D", "symmetric consensus selection", "machinery",
     "one fixed configuration per cohort x family x threshold, then a reviewed "
     "promotion",
     "one search per cohort; a complete matching record is loaded, never "
     "re-searched, and a non-matching one is refused"),
    ("D", "canonical source models",       "machinery",
     "one fit per family x threshold x seed, held for the rest of the run", "540 fits"),
    ("E", "the two references",            "primary analysis",
     "what the model achieves at home; what a local model reaches", "540 target-side fits"),
    ("E", "baseline ladder",               "primary analysis",
     "what survives the crossing", "no new fit"),
    ("F", "configuration comparison",      "primary analysis",
     "does the loaded configuration beat an untuned one", "1,080 fits, untuned arm only"),
    ("G", "label-free adaptation",         "primary analysis",
     "what is buyable with no target outcomes", "1 fit per cell"),
    ("H", "label-using adaptation, k=500", "primary analysis",
     "what five hundred target outcomes buy", "1-3 fits per cell"),
    ("I", "prior correction",              "primary analysis",
     "correcting the target rate with no target labels", "no new fit"),
    ("J", "label-budget curves",           "primary analysis",
     "where the returns to a label budget flatten", "expensive: 1 fit per budget"),
    ("K", "calibration correction",        "primary analysis",
     "splitting a budget between updating and calibrating", "expensive: 8 update fits per cell"),
    ("L", "structure transfer",            "exploratory follow-up",
     "does tree structure carry where parameters do not", "tree families only"),
    ("M", "leave one pillar out",          "robustness",
     "does one welfare pillar carry the result", "three families, shared partition"),
    ("M", "outcome variants",              "robustness",
     "does the result depend on the five-pillar construct", "three families, own partitions"),
]

display(pd.DataFrame(EXPERIMENTS,
                     columns=["section", "experiment", "status", "answers", "cost"])
        .set_index(["section", "experiment"]))

### Scope

What this run covers. Everything below reads these fields and nothing else sets them. Narrowing
`FAMILIES` narrows the run; it does not produce a reduced version of the analysis, because every
comparison below is defined over the families in scope and a subset answers a different question
from the one the manuscript reports.

Two switches turn expensive work on. Both are off, and both are scope decisions rather than
performance ones — with either off, the question it answers is undetermined rather than settled.

In [ ]:
# The nine families are `regime_names`' vocabulary rather than a list retyped here.
FAMILIES   = regime_names.FAMILIES
THRESHOLDS = (1, 2, 3)               # the ACE-count cuts this run reports
SEEDS      = range(20)               # the twenty-seed protocol; seed 0..19

# Rung 1 of Section C's baseline ladder: transfer with no target-cohort statistic in it at all.
# It is in no published table, and it is a further len(FAMILIES) x len(THRESHOLDS) x len(SEEDS)
# source fits. Section C says what turning it on would settle.
RUN_SOURCE_SCALED = False

# The nested per-split target redevelopment is NOT part of this notebook. It is a
# sensitivity analysis — a fresh YRBS selection inside each of the twenty training
# partitions — and its source function stays in `models`, tested, for a separate reading.
# Nothing here loads, runs, displays, hands off or publishes it.

# MCS counts below this are not printed. Section B says how the rule is applied and why the
# threshold counts need it. YRBS is open CDC data and is reported in full.
SUPPRESS_BELOW = 10

# The two things the lines above do not show: how much work this is, and where it lands.
# `work_path` raises here rather than partway through a section if $THESIS_WORK_DIR is unset.
_source_fits = len(FAMILIES) * len(THRESHOLDS) * len(SEEDS)
print(f"{_source_fits:,} canonical source fits  ->  tables land in {config.work_path()}")
print(f"baseline ladder rung 1 (source-scaled): {'ON' if RUN_SOURCE_SCALED else 'off'}")

---
# B — Data, outcomes, splits and what the split checks establish

Notebook 01 persists six cohort artefacts. This notebook reads four of them — the harmonised
features and the ACE pillars, per cohort. The two attribute frames are evaluation-only and
belong to notebooks 03 and 04. Nothing here re-reads a raw cohort: a second route from raw
would be a second definition of the features.

**The outcome** is a count of how many of the five ACE pillars a respondent reports, cut at one,
two, or three or more. It is derived from the persisted pillars rather than stored, so
`make_outcome` stays its only definition. It runs with `strict=True`, which is not exposed as a
parameter: `strict=False` fills missing pillars with zero, which turns "not asked" into "not
exposed" and inflates the YRBS numbers.

The prevalences below are the **no-skill PR-AUC nulls**. They must travel with every PR-AUC
reported anywhere, and this is the only place the pipeline shows them per threshold.

### The display rule for the MCS counts

Printing three nested counts beside their denominator discloses more than three figures. If
n(>=1), n(>=2), n(>=3) and the analytic-sample size are all on the line, then the number of
respondents reporting exactly one pillar is the first minus the second, exactly two is the
second minus the third, and none is the denominator minus the first. Four cells, exhaustive,
every one recoverable by subtraction from what was printed.

So the rule is applied to those four implied cells, not to the three figures on the line. If
every implied cell is zero or at least ten, the MCS block prints in full. If any implied cell
falls between one and nine, the whole MCS count-and-prevalence block is withheld — a partial
suppression would be undone by the figures that remain plus the denominator, which is the
complementary-disclosure route the rule exists to close.

Withholding the block would also withhold the MCS PR-AUC nulls, which every MCS PR-AUC needs.
If that happens, they have to be obtained through an authorised disclosure route rather than
printed here. **This is a display rule and nothing more**: the exact counts are computed in
memory and used, and no file records what was withheld.

YRBS is open CDC data and is reported in full. The rule is checked below on synthetic series.

In [ ]:
def implied_cells(counts, denominator):
    """The exhaustive partition that printing cumulative counts beside a denominator reveals.

    `counts` maps threshold -> number at or above it, `denominator` is the analytic sample.
    Returns one entry per recoverable cell, in ascending order of pillar count.
    """
    order = sorted(counts)
    cells = {f"below >={order[0]}": denominator - counts[order[0]]}
    for lo, hi in zip(order, order[1:]):
        label = f"exactly {lo}" if hi == lo + 1 else f"{lo} to {hi - 1}"
        cells[label] = counts[lo] - counts[hi]
    cells[f">={order[-1]}"] = counts[order[-1]]
    return cells


def mcs_block_is_printable(counts, denominator):
    """Whether the MCS threshold block may be printed, and which implied cells forbid it.

    A cell of exactly zero is not a disclosure and does not withhold anything, which is the
    same treatment notebook 01's display helper gives it.
    """
    cells = implied_cells(counts, denominator)
    small = sorted(k for k, v in cells.items() if 0 < v < SUPPRESS_BELOW)
    return (not small), small


# The rule, on synthetic counts. No cohort is touched by any line here.
print("display rule checked on synthetic counts")

In [ ]:
# The artefacts notebook 01 wrote. Reading only the declared predictors gives the canonical
# column order and drops anything the schema does not name.
#
# `data.read_harmonised_features` refuses a feature parquet built under a different
# `data.PREPROCESSING_VERSION`, or recording a different harmonised schema, so a frame left
# over from a run under an earlier recoding cannot reach a model here.
#
# `data.model_features` is the one transformation from the harmonised schema to the model
# schema, and `data.build_splits` refuses a frame that has not been through it.
X_mcs = data.model_features(
    data.read_harmonised_features(
        config.MCS_FEATURES,
        columns=list(data.FEATURE_COLUMNS),
    )
)
X_yrbs = data.model_features(
    data.read_harmonised_features(
        config.YRBS_FEATURES,
        columns=list(data.FEATURE_COLUMNS),
    )
)

pillars_mcs = pd.read_parquet(config.MCS_PILLARS)
pillars_yrbs = pd.read_parquet(config.YRBS_PILLARS)

# Composed here rather than stored, so `make_outcome` stays the outcome's one definition.
y_mcs = {t: data.make_outcome(pillars_mcs, t) for t in THRESHOLDS}
y_yrbs = {t: data.make_outcome(pillars_yrbs, t) for t in THRESHOLDS}

# Strict composition gives every threshold the same missing-value pattern, so one analytic
# sample size covers all three.
mcs_counts = {t: int(y_mcs[t].sum()) for t in THRESHOLDS}
mcs_analytic_n = int(y_mcs[THRESHOLDS[0]].notna().sum())
yrbs_analytic_n = int(y_yrbs[THRESHOLDS[0]].notna().sum())

print(
    f"YRBS {len(pillars_yrbs):,} rows, "
    f"{yrbs_analytic_n:,} with all five shared pillars"
)
for t in THRESHOLDS:
    print(
        f"  >={t}  prevalence {y_yrbs[t].mean():.4f} "
        f"({int(y_yrbs[t].sum()):,})"
    )

printable, blocking = mcs_block_is_printable(
    mcs_counts,
    mcs_analytic_n,
)

if printable:
    print(
        f"\nMCS  {len(pillars_mcs):,} rows, "
        f"{mcs_analytic_n:,} with all five shared pillars"
    )
    for t in THRESHOLDS:
        print(
            f"  >={t}  prevalence {y_mcs[t].mean():.4f} "
            f"({mcs_counts[t]:,})"
        )
else:
    print(
        "\nMCS  counts and prevalences not reported: the four thresholds are an "
        "exhaustive partition, so no part of the block can be shown without giving "
        "a withheld cell back by subtraction."
    )
    print(
        "     The MCS PR-AUC nulls therefore need an authorised disclosure route."
    )

# Each nominal predictor is replaced by one indicator per non-reference level.
# Derive the expected model-column count rather than writing it manually.
_contrasts = sum(
    len(levels) - 1
    for levels in data.NOMINAL_FEATURES.values()
)
_expected_model_columns = (
    len(data.FEATURE_COLUMNS)
    - len(data.NOMINAL_FEATURES)
    + _contrasts
)

if len(data.MODEL_FEATURE_COLUMNS) != _expected_model_columns:
    raise ValueError(
        "the model schema is not the harmonised schema with each declared "
        "nominal predictor replaced, in place, by its indicators"
    )

print(
    f"\nfeature schema  {len(data.FEATURE_COLUMNS)} harmonised predictors -> "
    f"{len(data.MODEL_FEATURE_COLUMNS)} model columns "
    f"({data.PREPROCESSING_VERSION})"
)

for name, levels in data.NOMINAL_FEATURES.items():
    indicators = [
        column
        for column in data.MODEL_FEATURE_COLUMNS
        if column.startswith(f"{name}_")
    ]
    print(
        f"  {name} enters as {', '.join(indicators)}, "
        f"against {list(levels)[0]}"
    )

# The raw-versus-analytic MCS difference is notebook 01's open disclosure question and is not
# settled here: nothing below prints or derives it.

**The split protocol.** Twenty seeds, a stratified 75/25 cut of each cohort, and a target-label
pool taken from the training half so it cannot intersect the test set. Every published
comparison is paired *within* these splits: a procedure and its baseline are read on the same
respondents under the same fitted models, so their difference is a within-seed quantity. That
pairing is what the twenty seeds buy.

**No inferential test is reported below.** The twenty splits are overlapping 75% draws from one
sample, sharing most of their training rows and about a quarter of each test set, so the
per-seed differences are not independent observations and the signed-rank test's assumptions are
not met. What is reported is the mean paired difference across the twenty shared splits and its
spread — descriptive quantities the design does support. Section N states the limitation.

The splits are rebuilt every run rather than persisted. They are deterministic given the
features, the outcomes and the seed, and they hold rows — storing them would multiply
restricted material on disk for nothing.

**This is where standardisation happens.** Notebook 01 persists both cohorts at raw scale and
standardises nothing. `data.build_splits`, called by the cell below, fits an imputer and a
scaler *inside* each split — five times per bundle, each on a different frame. That is why it
returns frames rather than index arrays, and Section C is about what it means.

`stratify=y.fillna(0)` counts a missing outcome as a negative **for the purpose of the cut
only**. It keeps the NaN-outcome rows spread across train and test instead of clustering, and
it happens before those rows are dropped from training.

In [ ]:
# Held for the notebook's life; every section below reads them. The eighteen frames in each
# bundle keep S4's names (`Xm_cs`, `yte`, `pool_cs`), which `SplitBundle` pins.
splits = {(seed, t): data.build_splits(seed, y_mcs[t], y_yrbs[t],
                                       X_mcs=X_mcs, X_yrbs=X_yrbs, threshold=t)
          for t in THRESHOLDS for seed in SEEDS}

print(f"{len(splits)} split bundles x {len(data.SPLIT_KEYS)} frames each")
print(splits[(min(SEEDS), THRESHOLDS[0])].shapes())      # shapes only — Tier 1

### What the split checks establish

Four assertions, and it matters what each one is entitled to claim. Three are about how the
draws were constructed within one cohort. The fourth compares index labels across cohorts and
is the weakest of the four — it is stated narrowly below rather than as a guarantee about
respondents.

One consequence is worth stating separately, because it is easy to misread. **The evaluable
sample is smaller than the test frame.** The split is taken on the full YRBS frame and only the
*training* side has its NaN-outcome rows dropped; the test frame keeps them and each metric drops
them at evaluation. So `n_test` reads well below the frame's row count. Both numbers are
correct.

In [ ]:
problems = []

for (seed, threshold), split in splits.items():
    pool_index = set(split["pool_cs"].index)
    test_index = set(split["Xy_te"].index)

    # The target-label pool and YRBS test set come from opposite sides
    # of the same YRBS split.
    if pool_index & test_index:
        problems.append((seed, threshold, "target-label pool intersects YRBS test set"))

    # The k=500 experiment must draw only from the target-label pool.
    anchor = split.k_slice(transfer.K)
    if not set(anchor.index).issubset(pool_index):
        problems.append((seed, threshold, "k=500 sample falls outside target-label pool"))

    # The budget analysis must begin with the same k=500 sample.
    budget_order = split.nested_draw()
    if not budget_order.index[:transfer.K].equals(anchor.index):
        problems.append((seed, threshold, "budget ordering does not begin with k=500 sample"))

if problems:
    raise ValueError(
        f"{len(problems)} split check(s) failed; first five: {problems[:5]}"
    )

example = splits[(min(SEEDS), THRESHOLDS[0])]
test_outcome = np.asarray(example["yte"], dtype=float)
n_evaluable = int(np.sum(~np.isnan(test_outcome)))

print(
    f"Checked {len(splits)} split bundles. The target-label pool is separate from "
    "the YRBS test set, the k=500 sample comes from that pool, and the budget "
    "analysis begins with the same sample."
)
print(
    f"Example split: {len(example['Xy_te']):,} YRBS test rows, including "
    f"{n_evaluable:,} with a defined outcome; target-label pool "
    f"{len(example['pool_cs']):,} rows."
)
print(
    "MCS and YRBS index labels are not compared because they come from separate "
    "files and are not cross-cohort respondent identifiers."
)

---
# C — Preprocessing: what cohort standardisation is doing

**Question.** How much of what the pipeline calls *unadapted transfer* is already an adaptation?

`data.standardise_cohort` fits its imputer and scaler on whichever frame it is handed, so every
`_cs` frame in a `SplitBundle` is standardised against **itself**, test frames included. The
YRBS test frame the source model is scored on has been re-centred on YRBS statistics before the
model sees it. That is a label-free adaptation, and §V-B counts it as one.

**It is applied here, not upstream.** Notebook 01 persists both cohorts at raw scale and
standardises nothing; there is no single standardised cohort to store, because the transform is
fitted per split. Section B's `build_splits` call is where it happens, and this notebook owns
the decision and its consequences.

Two consequences. The target-side transform is **transductive**: how a test row is standardised
depends on the other test rows in its frame, so a prediction is not a function of that
respondent's own features alone. And within one cohort both reference arms fit in training units
and predict in test units, because `Xy_te_cs2` is bound to `Xy_te_cs` — the `_cs`/`_cs2`
distinction is real on the train side and **empty on the test side**.

This is a modelling decision and a limitation, recorded in Section N. **The method is
fixed and this notebook does not change it.**

### The baseline ladder

Every published number in this notebook sits at rung 2 or above.

| rung | what it is | target information used |
|---|---|---|
| **1** `source_scaled` | the source model with its own imputer and scaler carried across, applied to raw target features | none |
| **2** `unadapted` | the same model, after both cohorts are standardised against themselves | the target cohort's own feature distribution |
| **3** label-free | rung 2 plus quantile mapping, importance weighting or pseudo-labelling | target features, still no outcomes |
| **4** label-using | rung 2 plus 500 target outcomes | target outcomes |

**Rung 1 is not in the published results.** Nothing under `outputs/` contains it. The code path
is `transfer.source_scaled_scores`; Section E runs it when `RUN_SOURCE_SCALED` is set, and does
not otherwise.

**So with the flag off, "unadapted" is a cohort-standardised, transductive baseline and not a
raw one.** Table headers say so — `SHORT_DISPLAY` spells it *unadapted (cohort-std)*. The
pipeline key is unchanged, because every frozen table and downstream reader is keyed on it.

**A plausible expectation to examine, not a recorded one.** A mean and scale mismatch translates
directly into a shifted linear predictor, so rung 1 might sit below rung 2 for the linear families
and closer to it for the boosted ones, whose split points are thresholds on raw values. Nothing
here measures it; the sign of the difference is what the rung would settle.

**Status: diagnostic.** The cell below reads the split frames and fits nothing.

In [ ]:
# The standardisation claim, checked on one bundle rather than asserted. Aggregates only.
S0 = splits[(min(SEEDS), THRESHOLDS[0])]

print(f"Xy_te_cs2 equals Xy_te_cs cell for cell : {S0['Xy_te_cs2'].equals(S0['Xy_te_cs'])}")
for key, what in (("Xm_cs", "MCS train (what models fit on)"),
                  ("Xm_te_cs", "MCS test  (source reference eval)"),
                  ("Xy_te_cs", "YRBS test (every transfer eval)")):
    f = S0[key]
    print(f"  {key:<10s} max |column mean| {f.mean().abs().max():.2e}   "
          f"max |sd - 1| {(f.std(ddof=0) - 1).abs().max():.2e}   <- {what}")
print("\nA frame centred on itself has column means at machine zero. Each of the three is,")
print("so each carries its own scaler rather than the training frame's.")

# How far apart the two cohorts are BEFORE any standardisation — the shift rung 2 removes.
_shift = (X_mcs.mean() - X_yrbs.mean()).abs()
print(f"\nraw cohort mean difference across {len(_shift)} features: "
      f"median {_shift.median():.3f}, max {_shift.max():.3f}")

---
# D — Model selection and canonical source models

**Question.** Under one procedure applied to each cohort separately, which configuration should
each model family be fitted under at each outcome threshold — and what does the resulting
welfare-risk model built on MCS look like?

Four movements: the selection itself, the review of what it chose, the reviewed promotion that
turns it into the tracked specification, and the canonical fits every experiment below reuses.

**Selection happens here; promotion does not.** The gate at the end of the third movement refuses
to fit anything until the promoted specification and the private records agree cell for cell.

**Status: machinery.** No result is claimed in this section. The cross-validated values in the
review table are internal selection diagnostics.

The four movements are the four sub-headings below.

## Symmetric consensus selection

**Question.** Under one procedure applied to each cohort separately, which configuration should
each model family be fitted under, at each outcome threshold?

This part of Section D **selects and saves. It does not promote.** Promotion into the tracked
specification is a separate, reviewed act — `scripts/promote_local_settings.py` — so that no
ordinary run can change the configuration every result below is conditional on.

| | |
|---|---|
| **development seeds** | 0, 1 and 2 |
| **data** | that cohort's outer-**training** partition for that seed and threshold, and nothing else |
| **inner folds** | five, stratified, shuffled on the development seed; built once per cohort, threshold and seed, and shared by every candidate |
| **preprocessing** | each fold half median-imputed and standardised **against itself** |
| **estimator** | `models.make_estimator`, the same factory the battery fits with |
| **objective** | AUC |
| **aggregation** | mean fold AUC within each development seed; then the mean of those three seed-level means, and their population SD (`ddof=0`) |
| **eligibility** | estimable on all three development seeds, or not a candidate |
| **tie rule** | highest three-seed mean; exact tie → lowest SD; still tied → candidate-pool order |
| **output** | one fixed configuration per (threshold, family) — 27 per cohort, 54 in total |

**A model uses the settings selected in the cohort it is trained on.** The MCS mapping configures
the MCS local reference, the canonical source models and every forward transfer and adaptation
regime. The YRBS mapping configures the YRBS local reference and the reverse YRBS-to-MCS transfer
model, and reaches no forward adaptation procedure.

**No outer test partition of either cohort is read here, and neither cohort's selection reads the
other's data.** That is a property of the interface rather than of care: `consensus_select` takes
prepared training frames and has no cohort concept at all, and the two adapters that build them
each read exactly two named keys.

**This is a replacement, not a reconstruction.** It does not reproduce, recover or approximate any
earlier specification, and nothing here should be read as evidence about one.

The protocol below is read from `models` rather than restated. A second copy of it in this
notebook would be a second source of truth, and the two would eventually disagree.

**Status: machinery, and searched once.** No result is claimed here. After the first run, a
complete record whose provenance is the live one is loaded rather than recomputed; a record that
does not match the live provenance is refused rather than re-searched, and waits for you to
inspect it and move or delete it. Nothing is topped up or silently rebuilt.

In [ ]:
# The declared protocol, read from `models`. FAMILIES and THRESHOLDS are Section A's bindings
# and are not re-declared here: narrowing the run's scope must narrow the selection with it.
DEV_SEEDS = models.DEV_SEEDS

if not set(int(s) for s in DEV_SEEDS) <= set(int(s) for s in SEEDS):
    raise ValueError(
        f"the development seeds {list(DEV_SEEDS)} are not all inside this run's evaluation "
        f"seeds. The selection reads the split bundles Section B already built; it does not "
        f"build its own.")

POOL_SIZES = {family: len(models.candidate_pool(family)) for family in FAMILIES}

print(f"development seeds   {list(DEV_SEEDS)}")
print(f"inner folds         {models.CV_FOLDS}, stratified, shuffled on the development seed")
print(f"objective           {models.OBJECTIVE}")
print(f"tie rule            {models.TIE_RULE}")
print(f"thresholds          {list(THRESHOLDS)}")
print(f"families            {len(FAMILIES)}: {', '.join(FAMILIES)}")
print(f"candidates          {sum(POOL_SIZES.values())} per threshold  "
      f"({', '.join(f'{f}={n}' for f, n in POOL_SIZES.items())})")
print(f"protocol_id         {models.protocol_id()}")
print(f"preprocessing       {data.PREPROCESSING_VERSION}")
print(f"\nsearch fits per cohort: "
      f"{sum(POOL_SIZES.values()) * models.CV_FOLDS * len(THRESHOLDS) * len(DEV_SEEDS):,}")
print(f"records land in {config.work_path()}")

### The development bundles

**Three seeds, three thresholds, nine bundles — taken out of the twenty-seed `splits` Section B
already built.** Nothing here rebuilds a feature frame, an outcome or a split: a second split
route would be a second definition of the partition every result below is measured on.

Each bundle carries both cohorts, so the same nine serve both selections; the adapters take the
MCS training frames out of them for one and the YRBS training frames for the other.

In [ ]:
# The two adapters. Each reads exactly two named keys out of each development bundle — MCS:
# Xm_trm, ym_trm; YRBS: Xy_trm, yy_trm — and returns frames. The selector never receives a
# bundle, so no outer test frame and no other-cohort frame is in its scope at all.
MCS_FRAMES = models.mcs_dev_frames(splits, thresholds=THRESHOLDS, dev_seeds=DEV_SEEDS)
YRBS_FRAMES = models.yrbs_dev_frames(splits, thresholds=THRESHOLDS, dev_seeds=DEV_SEEDS)

print(f"MCS  development frames: {len(MCS_FRAMES)} (threshold, seed) cells")
print(f"YRBS development frames: {len(YRBS_FRAMES)} (threshold, seed) cells")
print("outer-test frames are not among them, in either cohort")

### Both cohorts, one procedure

**Load a complete compatible record, or search from nothing. There is no third path.** A saved
record is used only if its provenance is the live one — the same preprocessing version, model
feature schema, development seeds, fold count, objective, tie rule, candidate pool and protocol
identifier. When no record exists for a cohort, one fresh search covers the whole grid and is
saved at once.

**A record that is incompatible, incomplete or stale is REFUSED, and the run stops there.** It is
not topped up, not silently replaced and not partly recomputed: half a grid searched under one
candidate pool and half under another is not one selection, and overwriting the file automatically
would destroy the only description of the configuration an existing result was fitted under. The
refused record stays where it is. Inspect it, **move or delete it deliberately**, and re-run — the
cell then finds no record and performs one fresh complete search.

**A cell with no candidate estimable on all three development seeds stops the run.** There is no
fallback, no dropped cell, no reduced fold count and no selection on fewer seeds — each of those
would put a configuration chosen under a different protocol into a column that says it was chosen
under this one.

The two calls below are the same call with the other cohort's training frames. Nothing about the
selection differs between them, which is what makes the resulting local references comparable.

In [ ]:
def load_or_select(cohort, frames):
    """Load a complete, compatible record; search only when there is none at all.

    THREE OUTCOMES, AND ONLY THREE.

      * a COMPLETE record whose provenance is the live one -> loaded; nothing is re-searched;
      * NO record for this cohort -> one fresh search over the whole grid, saved at once;
      * an INCOMPATIBLE, INCOMPLETE or STALE record -> `load_selection_record` raises and this
        stops. Only `FileNotFoundError` reaches a search; a `ValueError` is not caught here.

    NOTHING IS TOPPED UP, SILENTLY REPLACED OR PARTLY RECOMPUTED. A refused record is left
    exactly where it is. It is the evidence of what an earlier run selected, and an automatic
    overwrite would destroy the only description of the configuration some existing result was
    fitted under. Inspect it, move or delete it deliberately, then re-run this cell — which then
    finds no record and performs one fresh complete search.
    """
    try:
        models.load_selection_record(cohort=cohort)
        print(f"{cohort.upper()}: loaded the saved record; nothing was re-searched")
        return
    except FileNotFoundError:
        pass
    started = time.time()

    def report_progress(threshold):
        print(
            f"  >={threshold} selected  ({time.time() - started:.0f}s elapsed)",
            flush=True,
        )

    records = models.consensus_select(
        frames,
        families=FAMILIES,
        thresholds=THRESHOLDS,
        dev_seeds=DEV_SEEDS,
        progress=report_progress,
    )
    # SAVED IMMEDIATELY, before anything is displayed or read from it. This is the expensive
    # computation in the notebook and it is not checkpointed while it runs.
    written = models.save_selection_record(
        records, cohort=cohort, families=FAMILIES, thresholds=THRESHOLDS, dev_seeds=DEV_SEEDS)
    print(f"{cohort.upper()}: searched and saved {len(records)} selections to {written.name}")


load_or_select("mcs", MCS_FRAMES)

In [ ]:
load_or_select("yrbs", YRBS_FRAMES)

### The private records, read back

Both records are re-opened through the ordinary loader, so what is reviewed below is the file that
a promotion will read — not an in-memory object that happens to agree with it.

**The records stay under the working root.** They carry every candidate's three seed-level mean
AUCs, their mean and their standard deviation. The MCS values are MCS-derived and need separate
disclosure review; none of it is promoted, and none of it reaches the repository, a publication
candidate, a LaTeX fragment or a manuscript table.

In [ ]:
LOADED = {cohort: models.load_selection_record(cohort=cohort) for cohort in models.COHORTS}

for cohort, loaded in LOADED.items():
    models.selection_coverage(loaded["records"], families=FAMILIES, thresholds=THRESHOLDS)
    print(f"{cohort.upper()}: {len(loaded['records'])} cells, "
          f"{len(loaded['payload']['ranking'])} candidate records, "
          f"protocol {loaded['payload']['protocol_id']}")

## Selection review

Two tables, both read from the saved records.

The **coverage table** confirms the shape of the selection: 27 cells per cohort, every family at
every threshold, the expected fold and candidate counts, the status, and how the winner was
decided in each cell. It carries no respondent count, no path and no cohort quantity.

The **selected-configuration table** is the review itself: for every cell, all three development
seeds' mean AUCs, the three-seed mean and standard deviation, which clause of the tie rule decided
it, and the configuration that will be promoted. It is read from the record rather than
recomputed, because a recomputed table could disagree with the file the promotion reads.

**Both tables are always built and always checked; only their rendering is guarded.** The internal
run is the one a promotion decision is made from, and it shows them. The generated public copy
keeps no output from any cell in this section — none of them is in
`spec/public_notebook_cells.json`'s allow-list for this notebook — so the guard and the allow-list
say the same thing twice, deliberately.

**These are internal.** The cross-validated values in them reach no tracked specification, no
publication candidate, no LaTeX fragment and no manuscript table.

In [ ]:
coverage_rows = []
for cohort, loaded in LOADED.items():
    payload, records = loaded["payload"], loaded["records"]
    summary = models.selection_coverage(records, families=FAMILIES, thresholds=THRESHOLDS)
    for threshold in THRESHOLDS:
        cells = [records[(int(threshold), f)] for f in FAMILIES]
        coverage_rows.append(dict(
            cohort=cohort, threshold=f">={threshold}",
            families=len(FAMILIES),
            cells=len(cells),
            selected=sum(1 for c in cells if c["status"] == models.SELECTION_SELECTED),
            folds=sorted({c["folds"] for c in cells}),
            candidates=sum(c["candidates"] for c in cells),
            dev_seeds=payload["development_seeds"],
            tie_mean=sum(1 for c in cells if c["tie_broken_on"] == "mean"),
            tie_sd=sum(1 for c in cells if c["tie_broken_on"] == "sd"),
            tie_pool_order=sum(1 for c in cells if c["tie_broken_on"] == "pool_order")))

coverage = pd.DataFrame(coverage_rows)

# THE SHAPE CHECKS RUN IN BOTH MODES. Only the rendering above them is a presentation decision.
for cohort, loaded in LOADED.items():
    expected = len(THRESHOLDS) * len(FAMILIES)
    if len(loaded["records"]) != expected:
        raise ValueError(f"{cohort} carries {len(loaded['records'])} cells, expected {expected}")
    if sorted({c["candidates"] for c in loaded["records"].values()}) != \
            sorted(set(POOL_SIZES.values())):
        raise ValueError(f"{cohort}: the recorded candidate counts are not the pool sizes")

if not PUBLIC_NOTEBOOK:
    print("coverage and status — no respondent count, path or cohort quantity appears here")
    display(coverage.set_index(["cohort", "threshold"]))
    print(f"\n{sum(len(l['records']) for l in LOADED.values())} selected cells in total "
          f"({len(models.COHORTS)} cohorts x {len(THRESHOLDS)} thresholds x "
          f"{len(FAMILIES)} families)")

In [ ]:
# THE SELECTED-CONFIGURATION REVIEW TABLE. Built and validated on every run; the promotion
# decision is made from it. Read from the saved records — `selection_review_frame` takes the
# loaded payload and reformats it, and computes nothing.
review = pd.concat([models.selection_review_frame(LOADED[cohort])
                    for cohort in models.COHORTS], ignore_index=True)

if list(review.columns) != ["cohort", "threshold", "family", "seed_0_auc", "seed_1_auc",
                            "seed_2_auc", "mean_auc", "sd_auc", "tie_broken_on", "params"]:
    raise ValueError(f"the review table's columns have drifted: {list(review.columns)}")
if review[["seed_0_auc", "seed_1_auc", "seed_2_auc"]].isna().any().any():
    raise ValueError("a selected cell is missing one of its three development-seed scores")
if len(review) != len(models.COHORTS) * len(THRESHOLDS) * len(FAMILIES):
    raise ValueError(f"the review table carries {len(review)} rows, expected "
                     f"{len(models.COHORTS) * len(THRESHOLDS) * len(FAMILIES)}")

if not PUBLIC_NOTEBOOK:
    print("INTERNAL — selection diagnostics, MCS-derived in part. Not for release without review.")
    display(review.set_index(["cohort", "threshold", "family"]).round(4))

## Reviewed promotion

**Nothing above wrote a tracked specification, and nothing below will.** Promotion is a separate,
deliberate command, run after the review table has been read:

```bash
python scripts/promote_local_settings.py                     # dry run: the full diff, nothing written
python scripts/promote_local_settings.py --apply --confirm   # writes spec/local_model_settings.csv
```

The promoter refuses unless both records load cleanly, describe the same procedure, and cover all
27 cells each with a selected configuration. It writes **one** file — 54 rows,
`cohort,threshold,family,settings,protocol_id,settings_digest` — through a single atomic replace,
backs up the previous version under the working root, and promotes **no cross-validated value**.

**Two identifiers travel with the specification, and both are needed.** `protocol_id` identifies
the selection *procedure*; `settings_digest` identifies the 54 *configurations*. Two different
selections run under the same procedure share the first and differ in the second, so a matching
`protocol_id` alone is not evidence that two sets of results are comparable.

**The gate below stops the notebook until the two agree.** On a first run it refuses, because the
promoted file does not yet describe the selection just made — that refusal is the expected first
state, not a failure. Promote, then re-run **from the gate cell**: the records are on disk and
`splits` is in memory, so nothing above is recomputed and nothing is re-searched.

**After promoting, re-run notebooks 02, 03 and 04 in order.** Every result produced before the
promotion was fitted under a different configuration, and outputs from either side of it must not
be mixed.

In [ ]:
# THE PROMOTION GATE. Refuses to fit anything until the promoted specification and the two
# private records describe the same 54 configurations. `scripts/promote_local_settings.py`
# remains the only writer of spec/local_model_settings.csv; this cell writes nothing.
#
# NO DICTIONARY EQUALITY AND NO SECOND DIGEST. `models.compare_settings` is the one comparator.
# It is tolerance-aware, so a float that survived a CSV round trip is not reported as a change,
# and it is keyed on the union of both mappings, so a missing, extra or mis-keyed cell appears
# as a non-matching row rather than as a shorter frame that would quietly pass.
#
# THE COMPARISON RUNS IN BOTH MODES. It is a gate, not a display.
_DRY_RUN = "python scripts/promote_local_settings.py"
_APPLY = "python scripts/promote_local_settings.py --apply --confirm"

# `load_local_settings` refuses an absent file, a stale protocol_id and an edited settings_digest
# before any per-cell comparison, so each of those arrives with its own message.
_promoted = models.load_local_settings()
_expected_cells = len(THRESHOLDS) * len(FAMILIES)
_gate_failures = []

for _cohort in models.COHORTS:
    _from_record = models.settings_from_record(LOADED[_cohort]["records"])
    _comparison = models.compare_settings(_from_record, _promoted[_cohort])
    if len(_comparison) != _expected_cells:
        _gate_failures.append(
            f"{_cohort}: the comparison covers {len(_comparison)} cells, expected "
            f"{_expected_cells} ({len(THRESHOLDS)} thresholds x {len(FAMILIES)} families)")
    for _row in _comparison[~_comparison["matches"]].itertuples(index=False):
        _gate_failures.append(
            f"{_cohort} / {_row.family} / {_row.threshold}: selected "
            f"{_row.selected or '(absent from the record)'} != promoted "
            f"{_row.other or '(absent from the specification)'}")

if _gate_failures:
    raise RuntimeError(
        "the promoted specification does not describe this selection, so no model was fitted. "
        f"{len(_gate_failures)} mismatching cell(s):\n  "
        + "\n  ".join(_gate_failures)
        + "\n\nReview the table above, then promote:\n"
        + f"  {_DRY_RUN}\n"
        + f"  {_APPLY}\n"
        + "and re-run this notebook from this cell.")

print(f"gate passed: {len(models.COHORTS) * _expected_cells} cells; the promoted specification "
      f"agrees with both private records")
print(f"  protocol {models.protocol_id()}")

## Canonical source models

**Question.** Under the configuration Section D just fixed, what does a welfare-risk model built
on MCS look like — and which of the experiments below may reuse it?

One model per (family, threshold, seed), fitted here and held for the rest of the notebook.

| | |
|---|---|
| **training cohort** | MCS Sweep 6, the training side of each split (`Xm_cs`, `ym_trm`) |
| **families** | the nine in `regime_names.FAMILIES` |
| **thresholds** | >=1, >=2, >=3 |
| **split seeds** | twenty, 0 to 19 |
| **model seed** | fixed at `models.RANDOM_STATE` = 42, so seed variation enters only through the split |
| **hyperparameters** | the MCS half of `spec/local_model_settings.csv`, selected above and promoted after review |
| **tuning** | none below this point in the notebook |

**What reuses these, and what may not.** Sections E, G, H, I, J, K and L take a model out of
this dictionary wherever the procedure adapts an existing source model. Four kinds of work fit
new models instead, and in three of the four the refit is what the experiment measures:

| | |
|---|---|
| **fits new models, and must** | anything trained on a re-expressed or augmented source cohort (quantile mapping, importance weighting, pseudo-labelling), anything trained on target outcomes (target-only, full revision, both ensembles' target side), the target-trained reference, the per-budget models in Section J, the per-ratio update models in Section K, and both outcome-robustness batteries in Section M — a different outcome is a different model by definition |
| **fits a new model only because it is a different configuration** | Section F's untuned arm |
| **reuses these** | everything else |

**Restricted artefacts.** These are fitted on MCS training rows and held in memory only.
Nothing writes them to disk: a tree ensemble can leak training rows through its leaf structure.
The figure printed after fitting is **serialised size — a proxy for what the dictionary costs,
not a measurement of process memory**. If it is far above expectation, that is something to
report rather than a reason to start evicting or persisting.

**Status: machinery.** The models are the input to every result below; nothing is claimed here.

In [ ]:
# The two fixed mappings every model in this notebook is fitted under. LOADED FROM THE
# PROMOTED SPECIFICATION, not from the records above: the selection wrote private
# records and the gate checked that the promoted file agrees with them cell for cell,
# but what configures a fit is the tracked file. Each cohort's 27 configurations were
# chosen by the same consensus procedure — development seeds 0, 1 and 2, five-fold
# stratified inner cross-validation within each development seed's outer-training
# partition, AUC, the mean of the three seed-level means — inside that cohort's own
# training partitions, and promoted after review. A RESTART-AND-RUN-ALL DOES NOT
# RE-SEARCH. A complete record made under the live procedure is loaded; one that is
# incompatible, incomplete or stale is refused and left in place for inspection,
# never topped up or silently rebuilt. Re-selecting per run would turn the reported
# across-seed spread into a search artefact.
#
# A MODEL USES THE SETTINGS SELECTED IN THE COHORT IT IS TRAINED ON. `MCS_SETTINGS`
# configures the MCS local reference, the canonical source models and every forward
# transfer and adaptation regime. `YRBS_SETTINGS` configures the YRBS local reference and
# nothing else in this notebook; the reverse transfer model it also configures is
# notebook 04's.
MCS_SETTINGS = models.mcs_settings()
YRBS_SETTINGS = models.yrbs_settings()
print(f"{len(MCS_SETTINGS)} MCS and {len(YRBS_SETTINGS)} YRBS (threshold, family) "
      f"configurations loaded from spec/local_model_settings.csv")
print(f"  protocol {models.protocol_id()}")

In [ ]:
source_models = transfer.fit_source_models(
    splits=splits, tuned=MCS_SETTINGS, families=FAMILIES, thresholds=THRESHOLDS, seeds=SEEDS)

print(f"  keys are (family, threshold, seed), e.g. {sorted(source_models)[0]}")
if len(source_models) != len(FAMILIES) * len(THRESHOLDS) * len(SEEDS):
    raise ValueError("the source models do not cover every family, threshold and seed")

### How the experiments below are run

Every experiment is one cell, and every cell binds a dataframe named after it. Nothing
accumulates in a shared structure and nothing is written while the notebook runs: a comparison
concatenates the frames it needs, so the code says which results are being compared.

`run_procedure` is the mechanical part — the loop over families, thresholds and seeds, and the
per-cell checks. It chooses no experiment: the procedure, the family scope and the regimes it is
expected to emit are all given at the call site, and it fits nothing itself. Each procedure
either takes a canonical source model or fits its own, and each cell below says which.

**Person-level scores.** Every procedure produces one prediction per YRBS test respondent per
seed, and most are used to compute metrics and then discarded. Five frames are kept, each
narrowed at the cell that produces it: unadapted and the target-trained reference for all nine
families, and the four focal k = 500 pipelines raw and recalibrated. Those are what the
downstream fixed-capacity, ventile, subgroup and uncertainty work is computed from. Keeping
every regime for every family would persist tens of millions of rows of person-level material
that no analysis reads.

In [ ]:
# One procedure over the grid, returning its own dataframe. Nothing accumulated, nothing
# written: the frame this returns is the frame the comparisons below concatenate.


def _resolve_families(name, families):
    """The families a regime cell will run over: checked, then put back in canonical order.

    `None` means every family in scope. Anything else is checked against the canonical nine and
    then intersected with `FAMILIES`, so a narrowed scope still narrows. The order is taken from
    `regime_names.FAMILIES` rather than from the argument, because several call sites pass a
    `frozenset` and a set has no order to inherit.

    An empty selection raises rather than widening. An empty list quietly meaning "all nine" is
    the failure this exists to prevent, and a selection that is empty *after* intersection would
    make the cell produce nothing while every completeness check passed vacuously.
    """
    canonical = list(regime_names.FAMILIES)
    if families is None:
        chosen = list(FAMILIES)
    else:
        asked = list(families)
        if not asked:
            raise ValueError(f"{name}: families= is empty. Name the families this procedure is "
                             f"defined for; an empty list does not mean all of them.")
        unknown = [f for f in asked if f not in set(canonical)]
        if unknown:
            raise ValueError(f"{name}: unknown famil(ies) {unknown}. The canonical nine are "
                             f"{canonical}.")
        chosen = [f for f in canonical if f in set(asked) and f in set(FAMILIES)]
    if not chosen:
        raise ValueError(f"{name}: no family in scope. FAMILIES is {list(FAMILIES)} and this "
                         f"procedure is defined for {sorted(families or canonical)}; the two do "
                         f"not overlap, so the cell would produce nothing.")
    return chosen


def run_procedure(name, procedure, *, families=None, expect_regimes=None,
                  sources=None, source_arg="base", source_family=None,
                  target_params=None, keep_scores=False, **kw):
    """Run `procedure` over families x thresholds x seeds and return its metric frame.

    Returns one dataframe. With `keep_scores=True` it returns `(metrics, scores)` instead. The
    cells that ask for scores narrow them to their declared focal scope on the spot; this
    function selects nothing and knows nothing about which regimes are kept.

    `families` is the declared scope, given at the call site. A procedure that returns nothing
    for a family it was declared over is an error here and now, not a cell quietly logged as
    inapplicable.

    `sources` is the dict from `fit_source_models`. `source_arg` names the keyword the
    procedure takes it under, and `source_family` overrides which family's source model is
    wanted — L1_LR for the coefficient-transfer procedures, CatBoost for the cross-family
    ensemble, both of which transfer from a fixed source whatever family is under test.

    `target_params` is the FIXED YRBS mapping, keyed `(threshold, family)` — no seed. It is
    passed only by the reference cell and reaches the procedure under its own keyword. IT
    NEVER TOUCHES `params`: `params` configures the MCS-trained model and this configures the
    YRBS-trained one, and the one line below is the whole boundary. Because the mapping
    carries no seed, all twenty splits share one configuration.
    """
    families = _resolve_families(name, families)
    expect = set(expect_regimes or (name,))
    wanted = {(f, t, s) for f in families for t in THRESHOLDS for s in SEEDS}
    rows, kept, seen, started = [], [], set(), time.time()

    for t in THRESHOLDS:
        for fam in families:
            params, hsrc = models.select_params(fam, "tuned", t, MCS_SETTINGS)
            for seed in SEEDS:
                S = splits[(seed, t)]
                extra = {}
                if sources is not None:
                    extra[source_arg] = transfer.source_for(
                        sources, source_family or fam, t, seed)
                if target_params is not None:
                    # A separate keyword, and `params` above is untouched. The YRBS-trained
                    # reference has to ask for its configuration by name, and the mapping it
                    # gets carries no seed, so all twenty splits share one configuration.
                    extra["target_params"] = target_params[(t, fam)]
                sc = procedure(S, family=fam, params=params, seed=seed, arm="tuned",
                               threshold=t, tuned=MCS_SETTINGS, **extra, **kw)
                # Checked inside the cell, not across the finished frame: a regime present for
                # nineteen seeds and missing for one is invisible in the completed dataframe.
                if set(sc) != expect:
                    raise RuntimeError(
                        f"{name} at {(fam, t, seed)} emitted {sorted(sc)}, expected "
                        f"{sorted(expect)}. It was declared applicable to {fam}; investigate, "
                        f"or narrow families= before running.")
                r, f, mismatched = transfer.metric_rows(
                    sc, S, family=fam, arm="tuned", threshold=t, seed=seed, source=hsrc,
                    keep_scores=keep_scores)
                if len(r) != len(expect):
                    raise RuntimeError(f"{name} at {(fam, t, seed)}: {len(r)} metric row(s) "
                                       f"for {len(expect)} regime(s)")
                # Scores that cannot be indexed by the YRBS test index mean the procedure
                # returned a vector of the wrong length, which is a defect whether or not this
                # run keeps the scores.
                if mismatched:
                    raise RuntimeError(
                        f"{name} at {(fam, t, seed)}: score rows do not match the YRBS test "
                        f"index ({mismatched}).")
                rows += r
                kept += f
                seen.add((fam, t, seed))

    if seen != wanted:
        raise RuntimeError(f"{name}: {len(wanted - seen)} declared cell(s) produced nothing, "
                           f"e.g. {sorted(wanted - seen)[:4]}")
    out = pd.DataFrame(rows)
    # Progress rather than result: it says how the run went, not what it found.
    if not PUBLIC_NOTEBOOK:
        print(f"{name}: {len(out):,} rows | {len(families)} families x "
              f"{len(THRESHOLDS)} thresholds x {len(SEEDS)} seeds | "
              f"{time.time() - started:.0f}s", flush=True)
    if keep_scores:
        return out, pd.concat(kept, ignore_index=True)
    return out


def summarise(frame):
    """Across-seed summary with the two reference gaps, for one explicitly built frame."""
    return evaluation.add_reference_gaps(evaluation.summarise_seeds(frame))


def compare_with_references(rows):
    """Compare procedures using mean and SD across the twenty splits.

    The target anchor is the YRBS LOCAL REFERENCE: a model developed and evaluated inside
    YRBS under the YRBS consensus selection. It takes one fixed configuration for all
    twenty splits, so it is complete by construction and there is no completeness rule to
    apply — the rule that used to live here belonged to a per-split search this notebook
    no longer runs.
    """
    threshold = HEADLINE_THRESHOLD

    shown = rows.loc[rows["threshold"] == threshold].copy()
    source = references.loc[
        (references["threshold"] == threshold)
        & (references["regime"] == "mcs_internal")
    ]
    target = references.loc[
        (references["threshold"] == threshold)
        & (references["regime"] == "yrbs_local")
    ]
    baseline = unadapted.loc[
        unadapted["threshold"] == threshold
    ]

    if shown.empty or source.empty or target.empty or baseline.empty:
        raise ValueError(
            f"One or more comparison frames are empty at {threshold}"
        )

    def metric_summary(frame, group_columns, prefix):
        return (
            frame.groupby(group_columns, observed=True)
            .agg(
                **{
                    f"{prefix}_auc": ("auc", "mean"),
                    f"{prefix}_auc_sd": ("auc", "std"),
                    f"{prefix}_prauc": ("prauc", "mean"),
                    f"{prefix}_prauc_sd": ("prauc", "std"),
                }
            )
            .reset_index()
        )

    procedures = metric_summary(
        shown,
        ["regime", "family"],
        "procedure",
    )
    source_summary = metric_summary(
        source,
        ["family"],
        "source_reference",
    )
    baseline_summary = metric_summary(
        baseline,
        ["family"],
        "unadapted",
    )
    target_local_summary = metric_summary(
        target,
        ["family"],
        "target_local",
    )

    # Changes are calculated within each matching split before being summarised.
    paired = shown[
        ["regime", "family", "seed", "auc", "prauc"]
    ].merge(
        baseline[["family", "seed", "auc", "prauc"]],
        on=["family", "seed"],
        suffixes=("_procedure", "_unadapted"),
    )

    paired["auc_change"] = (
        paired["auc_procedure"] - paired["auc_unadapted"]
    )
    paired["prauc_change"] = (
        paired["prauc_procedure"] - paired["prauc_unadapted"]
    )

    changes = (
        paired.groupby(["regime", "family"], observed=True)
        .agg(
            auc_change_from_unadapted=("auc_change", "mean"),
            auc_change_from_unadapted_sd=("auc_change", "std"),
            prauc_change_from_unadapted=("prauc_change", "mean"),
            prauc_change_from_unadapted_sd=("prauc_change", "std"),
        )
        .reset_index()
    )

    prauc_null = (
        baseline.groupby("family", observed=True)
        .agg(
            prauc_null=("prevalence_null", "mean"),
            prauc_null_sd=("prevalence_null", "std"),
        )
        .reset_index()
    )

    table = (
        procedures
        .merge(changes, on=["regime", "family"])
        .merge(source_summary, on="family")
        .merge(baseline_summary, on="family")
        .merge(target_local_summary, on="family", how="left")
        .merge(prauc_null, on="family")
    )

    table["auc_null"] = 0.5

    family_order = {
        family: position
        for position, family in enumerate(regime_names.FAMILIES)
    }
    regime_order = {
        regime: position
        for position, regime in enumerate(
            shown["regime"].drop_duplicates()
        )
    }

    table["_family_order"] = table["family"].map(family_order)
    table["_regime_order"] = table["regime"].map(regime_order)
    table = table.sort_values(["_regime_order", "_family_order"])

    columns = [
        "regime",
        "family",
        "auc_null",
        "source_reference_auc",
        "source_reference_auc_sd",
        "unadapted_auc",
        "unadapted_auc_sd",
        "procedure_auc",
        "procedure_auc_sd",
        "auc_change_from_unadapted",
        "auc_change_from_unadapted_sd",
        "target_local_auc",
        "target_local_auc_sd",
        "prauc_null",
        "prauc_null_sd",
        "source_reference_prauc",
        "source_reference_prauc_sd",
        "unadapted_prauc",
        "unadapted_prauc_sd",
        "procedure_prauc",
        "procedure_prauc_sd",
        "prauc_change_from_unadapted",
        "prauc_change_from_unadapted_sd",
        "target_local_prauc",
        "target_local_prauc_sd",
    ]

    out = (
        table[columns]
        .set_index(["regime", "family"])
        .round(4)
    )
    # No completeness attribute travels with the frame: the target local reference takes
    # one fixed configuration and is estimated on every split by construction.
    return out

def method_grid(frame, keys, metric):
    """Families down, procedures across, on one metric. Built fresh at each call site.

    Public mode shows the headline threshold; internal mode shows all three.
    """
    tables = {k: scores.method_table(frame, k) for k in keys}
    grid = pd.DataFrame(
        {regime_names.paper_label(k): t.set_index(["family", "threshold"])[metric]
         for k, t in tables.items() if not t.empty}).round(4)
    if PUBLIC_NOTEBOOK and "threshold" in grid.index.names:
        grid = grid.xs(HEADLINE_THRESHOLD, level="threshold", drop_level=False)
    return grid

def calibration_grid(frame, keys):
    """Calibration summaries across the twenty predefined splits."""
    metrics = [
        ("Calibration intercept mean", "cal_intercept_mean"),
        ("Calibration intercept SD", "cal_intercept_sd"),
        ("Calibration slope mean", "cal_slope_mean"),
        ("Calibration slope SD", "cal_slope_sd"),
        ("Brier mean", "brier_mean"),
        ("Brier SD", "brier_sd"),
        ("ECE mean", "ece_mean"),
        ("ECE SD", "ece_sd"),
    ]

    tables = {}
    for heading, metric in metrics:
        tables[heading] = method_grid(frame, keys, metric)

    return pd.concat(tables, axis=1)

---
# E — The two references and the main transfer baseline

**Question.** How much of a model built on MCS survives being applied to YRBS, and does
discrimination survive on the same terms as calibration?

Two references bound the answer, and neither crosses a border. The **source reference** is a
canonical source model on held-out MCS records — what it achieves at home. The
**target-trained reference** is a model fitted on the labelled YRBS pool — what a US authority
could reach building its own. The target-trained side is the one fit in this section that is
not a canonical model, and it has to be: that fit *is* the reference.

A plausible expectation to examine is that transfer lands between the two. That is a claim
about this data rather than a property of the arrangement: nothing forces a transferred model
to beat chance, and nothing forces the target-trained reference to beat the source one.

The section also asks whether discrimination and calibration degrade together. They measure
different things and only one is invariant to a monotone rescaling of the scores, so they can
come apart.

**Status: primary analysis.**

### Three reference points, and what each one is for

None of them crosses a border, and **none of them is a ceiling** — each is a model fitted
somewhere, and a transfer procedure that beats one is a result rather than a contradiction.

| reference | trained on | configuration | what its distance from unadapted transfer measures |
|---|---|---|---|
| **source reference** | MCS training half | the loaded settings | — (it is the other side of the border) |
| **target-trained reference** | YRBS training half | **the same loaded settings** | changing the training cohort, configuration held constant |
| **resource-rich target benchmark** | YRBS training half | **chosen inside this seed's own YRBS training partition** | changing the training cohort *and* having the resource to select a configuration |

The third is new here, and it is what makes the question in the title answerable: *what can a
well-resourced YRBS development procedure achieve, and how much of that does transfer recover
with five hundred labels?* The second alone cannot answer it, because it inherits a
configuration chosen on the wrong cohort.

**The information boundary.** The target-side configurations reach the benchmark and nothing
else. No transfer procedure, no other reference and no sensitivity battery receives them —
`run_procedure` passes them under their own keywords and never as `params`.

**Preprocessing is the same for all three.** Every YRBS frame in this pipeline is standardised
against itself, test frames included (Section C), and the benchmark is evaluated under that same
convention. So YRBS test *outcomes* enter neither the selection nor any fit, while YRBS test
*features* do enter their own cohort standardisation — as they do for unadapted transfer and for
every other target-side regime. The benchmark differs from the target-trained reference through
target-side hyperparameter selection, and through nothing else.

In [ ]:
references, reference_scores = run_procedure(
    "reference", transfer.reference_scores,
    expect_regimes=("mcs_internal", "yrbs_local"),
    sources=source_models, target_params=YRBS_SETTINGS, keep_scores=True)

# `metric_rows` excludes `mcs_internal` from every score frame at source, because those
# predictions are MCS row-level. Asserted here rather than trusted: this is the one procedure
# that scores both cohorts, so it is the one place the exclusion could matter.
_score_regimes = set(reference_scores["regime"])
if "mcs_internal" in _score_regimes:
    raise ValueError("an MCS-evaluated regime reached the score frame")
if not _score_regimes <= {"yrbs_local"}:
    raise ValueError(f"unexpected regime(s) in the reference score frame: "
                     f"{sorted(_score_regimes - {'yrbs_local'})}")

# THE LABEL IS PER REGIME. One of these two rows was fitted under the MCS configuration and one
# under the YRBS configuration, so a single label for the call would be wrong about one of them.
_sources = (references.drop_duplicates(["regime", "hyperparameter_source"])
            .set_index("regime")["hyperparameter_source"])
if _sources.index.has_duplicates:
    raise ValueError("a regime carries more than one hyperparameter source")
if _sources.get("yrbs_local") != models.YRBS_HYPERPARAMETER_SOURCE:
    raise ValueError("the YRBS local reference does not carry the YRBS configuration label")
if set(_sources.drop("yrbs_local", errors="ignore")) != {models.MCS_HYPERPARAMETER_SOURCE}:
    raise ValueError("a reference other than yrbs_local carries the YRBS label")

reference_labels = {
    "mcs_internal": "MCS-developed, tested on MCS",
    "yrbs_local": "YRBS-developed, tested on YRBS",
}

# Both local references are public anchors. There is no third reference to withhold:
# the cross-configured arm the design once carried is retired, and the nested per-split
# sensitivity is off by default and anchors nothing.
PUBLIC_REFERENCES = ["mcs_internal", "yrbs_local"]

reference_rows = references.loc[
    references["threshold"] == HEADLINE_THRESHOLD
].copy()
if PUBLIC_NOTEBOOK:
    reference_rows = reference_rows[
        reference_rows["regime"].isin(PUBLIC_REFERENCES)]

# No completeness rule applies here. Both local references take one fixed configuration
# from the tracked specification, so every (family, threshold) cell is estimated on every
# split by construction; the rule that used to stand here belonged to a per-split search.

reference_rows["reference"] = reference_rows["regime"].map(reference_labels)

reference_table = (
    reference_rows
    .groupby(["family", "reference"], observed=True)
    .agg(
        auc_mean=("auc", "mean"),
        auc_sd=("auc", "std"),
        prauc_mean=("prauc", "mean"),
        prauc_sd=("prauc", "std"),
    )
    .unstack("reference")
    .reindex([
        family
        for family in regime_names.FAMILIES
        if family in set(reference_rows["family"])
    ])
    .round(4)
)

display(reference_table)

if not PUBLIC_NOTEBOOK:
    print("\nhyperparameter source by regime")
    display(_sources.rename("hyperparameter_source"))

if not PUBLIC_NOTEBOOK:
    print(f"person-level scores retained: {len(reference_scores):,} rows over "
          f"{sorted(_score_regimes)}")

### Rung 2 — the cohort-standardised baseline

The canonical source model applied to the YRBS test frame unchanged. No target outcomes, no
correction, no refit. It is **not** a raw-scale transfer: both frames were standardised against
themselves first, which is Section C's rung 2, and every table spells it *unadapted
(cohort-std)* for that reason.

In [ ]:
# Reuses a canonical source model; fits nothing. Its per-person predictions are kept for
# all nine families: the fixed-capacity, ventile and subgroup analyses downstream are all
# computed from them.
unadapted, unadapted_scores = run_procedure(
    "unadapted", transfer.unadapted_scores, sources=source_models, keep_scores=True)
display(compare_with_references(unadapted))
if not PUBLIC_NOTEBOOK:
    print(f"person-level scores retained: {len(unadapted_scores):,} rows "
          f"({unadapted_scores.regime.nunique()} regime)")

### Rung 1 — transfer with no target-cohort statistic at all

Runs only when `RUN_SOURCE_SCALED` is set, and it is off by default. It fits its own raw-scale
pipeline rather than reusing a canonical source model, because the canonical models are fitted
on standardised frames and rung 1 is defined on raw ones.

In [ ]:
if RUN_SOURCE_SCALED:
    source_scaled = run_procedure("source_scaled", transfer.source_scaled_scores)
    display(compare_with_references(source_scaled))
else:
    source_scaled = None
    print("rung 1 not run (RUN_SOURCE_SCALED = False). With it disabled the contribution of")
    print("cohort standardisation to every number below is undetermined, not zero.")

### The baseline frame

The three anchors, concatenated explicitly. Every comparison in G, H and I reads this frame's
`unadapted` rows as its baseline.

In [ ]:
transfer_baseline = pd.concat(
    [references, unadapted] + ([source_scaled] if source_scaled is not None else []),
    ignore_index=True)
baseline_summary = summarise(transfer_baseline)

ANCHORS = ["mcs_internal", "unadapted", "yrbs_local"]
if source_scaled is not None:
    ANCHORS.insert(1, "source_scaled")

# The fixed-configuration YRBS reference remains available for the gap quantities in
# the next cell. BOTH local references are public anchors: the cross-configured arm
# that used to be withheld is retired, so there is no third reference to narrow to.
display_anchors = ANCHORS

print("Anchor performance across twenty predefined splits")

anchor_display = pd.concat(
    {
        "AUC mean": method_grid(
            transfer_baseline, display_anchors, "auc_mean"
        ),
        "AUC SD": method_grid(
            transfer_baseline, display_anchors, "auc_sd"
        ),
        "PR-AUC mean": method_grid(
            transfer_baseline, display_anchors, "prauc_mean"
        ),
        "PR-AUC SD": method_grid(
            transfer_baseline, display_anchors, "prauc_sd"
        ),
        "PR-AUC null": method_grid(
            transfer_baseline, display_anchors, "prevalence"
        ),
    },
    axis=1,
)

display(anchor_display)

CALIBRATION_ANCHORS = [
    "mcs_internal",
    "unadapted",
    "yrbs_local",
]

if source_scaled is not None:
    CALIBRATION_ANCHORS.insert(1, "source_scaled")

print("\nCalibration of the baseline and reference models")
display(
    calibration_grid(
        transfer_baseline,
        CALIBRATION_ANCHORS,
    )
)

Calibration is assessed separately from discrimination. A calibration intercept
of zero and slope of one are ideal, while lower Brier and ECE values are better.
The standard deviations describe variation across the twenty predefined
train/test splits; they are not confidence intervals. Because the Brier score
depends partly on outcome prevalence, comparisons between MCS and YRBS are
interpreted cautiously.


In [ ]:
# This table separates two questions:
# 1. How much discrimination changes when an MCS model is applied to YRBS.
# 2. What a model developed inside YRBS reaches on the same records.
#
# There is no third question about local selection any more: both cohorts' models are
# configured by the same procedure inside their own training partitions, so the
# development budget is not a difference between them.
#
# These are AUC contrasts. PR-AUC and the SD across splits are shown in the
# preceding anchor-performance table. Uncertainty in the contrasts is handled
# through the paired bootstrap analysis in notebook 03.

anchor_rows = baseline_summary.loc[
    baseline_summary["regime"].isin(ANCHORS)
]

anchors = (
    anchor_rows
    .pivot_table(
        index=["threshold", "family"],
        columns="regime",
        values="auc_mean",
    )
    .rename(
        columns={
            "mcs_internal": "source",
            "yrbs_local": "target_local",
        }
    )
)

gaps = (
    baseline_summary.loc[
        baseline_summary["regime"] == "unadapted"
    ]
    .set_index(["threshold", "family"])[
        [
            "transfer_loss",
            "target_resource_gap",
            "target_gap_reason",
        ]
    ]
    .rename(
        columns={
            "transfer_loss": "gap_to_source",
            "target_resource_gap": "gap_to_target_local",
        }
    )
)

anchor_columns = [
    "source",
    "unadapted",
    "target_local",
]

if source_scaled is not None:
    anchor_columns.insert(1, "source_scaled")

transfer_gap = (
    anchors[anchor_columns]
    .join(gaps)
    .reset_index()
)

transfer_gap["model_class"] = transfer_gap["family"].map(
    regime_names.FAMILY_CLASS
)

if transfer_gap["model_class"].isna().any():
    missing_families = sorted(
        transfer_gap.loc[
            transfer_gap["model_class"].isna(),
            "family",
        ]
        .astype(str)
        .unique()
    )
    raise ValueError(
        f"Model class is not defined for: {missing_families}"
    )

# The decomposition into "what the training cohort buys" and "what local selection
# buys" is gone with the cross-configured arm: there is one target local reference and
# one distance to it.
transfer_gap = transfer_gap[
    ["threshold", "family", "model_class"]
    + anchor_columns
    + [
        "gap_to_source",
        "gap_to_target_local",
        "target_gap_reason",
    ]
]

transfer_gap_display = transfer_gap

if PUBLIC_NOTEBOOK:
    # Both local references are public anchors, so nothing is withheld here beyond the
    # narrowing to the headline threshold.
    transfer_gap_display = transfer_gap_display.loc[
        transfer_gap_display["threshold"] == HEADLINE_THRESHOLD
    ]

display(transfer_gap_display.round(4))

if not PUBLIC_NOTEBOOK:
    unavailable = transfer_gap_display.loc[
        transfer_gap_display["target_gap_reason"]
        .fillna("")
        .ne("")
    ]
    if not unavailable.empty:
        print(
            f"{len(unavailable)} comparison cell(s) do not have a "
            "target resource gap:"
        )
        display(
            unavailable[
                ["threshold", "family", "target_gap_reason"]
            ]
        )

In [ ]:
# Where unadapted transfer sits relative to the two local references, computed from what
# this run produced. Counts of cells, not a share of anything.
#
# No chance-anchored attainment figure is reported. Recovery is measured from unadapted
# transfer to the YRBS local reference.
_lo = transfer_gap[["source", "target_local"]].min(axis=1)
_hi = transfer_gap[["source", "target_local"]].max(axis=1)
_n = len(transfer_gap)
print(f"transfer between the two local references : "
      f"{int(((transfer_gap['unadapted'] >= _lo) & (transfer_gap['unadapted'] <= _hi)).sum())} of {_n} cells")
print(f"MCS local reference above transfer        : {int((transfer_gap['gap_to_source'] > 0).sum())} of {_n}")
print(f"YRBS local reference above transfer       : {int((transfer_gap['gap_to_target_local'] > 0).sum())} of {_n}")
print(f"transfer above chance                     : {int((transfer_gap['unadapted'] > 0.5).sum())} of {_n}")
_available = int(transfer_gap["gap_to_target_local"].notna().sum())
print(f"target resource gap available             : {_available} of {_n} cells")

In [ ]:
panel_gap = transfer_gap[transfer_gap["threshold"] == ">=2"].set_index("family").reindex(
    [f for f in regime_names.FAMILIES if f in set(transfer_gap["family"])])
transfer_figure, ax = plt.subplots(figsize=(7.2, 3.0))
x = np.arange(len(panel_gap))
ax.plot(x, panel_gap["source"], "s--", color="#0A2540", ms=4, lw=0.9,
        label="source reference (MCS)")
# The YRBS local reference: the figure shows the same target anchor the tables do.
ax.plot(x, panel_gap["target_local"], "^--", color="#C69D00", ms=4, lw=0.9,
        label="YRBS local reference")
for i in range(len(panel_gap)):
    ax.plot([i, i], [panel_gap["unadapted"].iloc[i], panel_gap["source"].iloc[i]],
            color="#BBBBBB", lw=4, alpha=.5, zorder=0, solid_capstyle="round")
ax.plot(x, panel_gap["unadapted"], "o-", color="#01418F", ms=5, lw=1.4,
        label=regime_names.SHORT_DISPLAY["unadapted"])
if source_scaled is not None:
    ax.plot(x, panel_gap["source_scaled"], "d:", color="#7A3E9D", ms=4, lw=1.0,
            label=regime_names.SHORT_DISPLAY["source_scaled"])
ax.set_xticks(x); ax.set_xticklabels(panel_gap.index, rotation=30, ha="right")
ax.set_ylabel("AUC"); ax.set_title("What survives the crossing, >=2")
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.42))
plt.show()

### Discrimination against calibration

The source reference against the baseline, on AUC and on calibration slope. A slope of 1 is the
reference line; nothing that leaves the ranking untouched can move AUC, and nothing that leaves
the probabilities untouched can move the slope.

In [ ]:
calibration_panel = evaluation.calibration_summary(baseline_summary)
panel_cal = calibration_panel[calibration_panel["threshold"] == ">=2"]
calibration_figure, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
for ax, (col, label, ref) in zip(axes, [("auc_mean", "AUC", None),
                                        ("cal_slope_mean", "calibration slope", 1.0)]):
    for reg, marker in ((regime_names.SHORT_DISPLAY["mcs_internal"], "s"),
                        (regime_names.SHORT_DISPLAY["unadapted"], "o")):
        grp = panel_cal[panel_cal["regime"] == reg].set_index("family").reindex(
            [f for f in regime_names.FAMILIES if f in set(panel_cal["family"])])
        ax.plot(range(len(grp)), grp[col], marker=marker, ms=4, lw=1.0, label=reg)
    if ref is not None:
        ax.axhline(ref, color="#999999", ls=":", lw=0.8)
    ax.set_xticks(range(len(regime_names.FAMILIES)))
    ax.set_xticklabels(regime_names.FAMILIES, rotation=45, ha="right")
    ax.set_ylabel(label)
axes[0].legend(frameon=False)
calibration_figure.suptitle("Discrimination and calibration after transfer  (>=2)", y=1.02)
plt.show()

In [ ]:
_c = calibration_panel.pivot_table(index=["threshold", "family"], columns="regime",
                                   values=["auc_mean", "cal_slope_mean", "ece_mean"])
_src, _una = regime_names.SHORT_DISPLAY["mcs_internal"], regime_names.SHORT_DISPLAY["unadapted"]
_keep = (_c[("auc_mean", _una)] - 0.5) / (_c[("auc_mean", _src)] - 0.5)
_ss, _su = (_c[("cal_slope_mean", _src)] - 1).abs(), (_c[("cal_slope_mean", _una)] - 1).abs()
print(f"share of above-chance AUC retained: median {_keep.median():.3f}   "
      f"range {_keep.min():.3f} to {_keep.max():.3f}")
print(f"|calibration slope - 1|: source {_ss.median():.3f}   after transfer {_su.median():.3f}   "
      f"worse in {int((_su > _ss).sum())} of {len(_su)} cells")
print(f"ECE: source {_c[('ece_mean', _src)].median():.4f}   "
      f"after transfer {_c[('ece_mean', _una)].median():.4f}")

### Interpretation

Where a large share of the above-chance AUC is retained while the calibration slope moves away
from 1 and ECE rises, this may preserve ranking utility, while the probabilities require
recalibration before probability-based interpretation. Nothing about operational or clinical
usefulness follows from AUC and calibration alone.

Sections G and H ask how much of either is buyable, with no target outcomes and with five
hundred. Section I asks whether the probabilities can be corrected without disturbing the
ranking.

---
# F — Did the loaded configuration matter?

**Question.** Sections D and E ran under the configurations `spec/local_model_settings.csv`
holds. Would an untuned model have done as well — inside each cohort, and after transfer?

**Why it is here rather than earlier.** The tuned arm of this comparison is not a separate
experiment: it is the same estimator, the same hyperparameters, the same model seed and the
same training frames as Section D's canonical models, so its three numbers are Section D's and
Section E's. Reading them off those sections rather than refitting removes 1,080 fits and the
assertion that would otherwise be needed to check that two identical computations agree. The
untuned arm is the only new fitting here.

**What the comparison can and cannot settle.** The MCS configuration was selected on MCS
cross-validated AUC, across three development seeds, so the tuned arm should look good inside
MCS largely by construction. The question is whether it also helps *after* transfer, which the
selection does not imply either way.

**And it must not be read backwards.** The tuned arm is used throughout because it is the locked
specification. This comparison **evaluates** that choice; it is not the reason for it, and could
not be: each cohort's configuration was chosen inside that cohort's own training partitions, and
neither selection read the other cohort's data or any outer test partition.

**Status: primary analysis** for the within-cohort numbers; **diagnostic** for the arm
comparison after transfer.

In [ ]:
# The untuned arm: two fits per (threshold, seed, family) — one inside MCS, one inside
# YRBS. BOTH TAKE THE UNTUNED CONFIGURATION, which is the library default or the frozen
# one, not a selected configuration. The YRBS row is named `yrbs_local` because that is
# the role it stands in for; it does NOT use the YRBS consensus selection, and this arm
# therefore says nothing about what either cohort's selection was worth.
# Nothing holds a fitted model; the three anchors are scored and the model discarded.
_untuned_cells = list(product(THRESHOLDS, SEEDS, FAMILIES))
_rows = []

for i, (t, seed, fam) in enumerate(_untuned_cells, 1):
    S = splits[(seed, t)]
    params, source = models.select_params(fam, "untuned", t, MCS_SETTINGS)

    mcs = models.make_estimator(fam, params, seed=seed)
    mcs.fit(S["Xm_cs"], S["ym_trm"].astype(int))           # trained inside MCS
    yrbs = models.make_estimator(fam, params, seed=seed)
    yrbs.fit(S["Xy_tr_cs2"], S["yy_trm"].astype(int))      # trained inside YRBS

    for name, y, p in (("mcs_internal",  S["ymte"], mcs.predict_proba(S["Xm_te_cs"])[:, 1]),
                       ("unadapted",     S["yte"],  mcs.predict_proba(S["Xy_te_cs"])[:, 1]),
                       ("yrbs_local",    S["yte"],  yrbs.predict_proba(S["Xy_te_cs2"])[:, 1])):
        _rows.append(dict(family=fam, arm="untuned", hyperparameter_source=source, seed=seed,
                          threshold=f">={t}", eval_config=name,
                          **evaluation.metrics(y, p, prevalence=float(np.nanmean(y)))))
    print(f"  {i}/{len(_untuned_cells)} untuned cells", end="\r", flush=True)

# The tuned arm, read off the frames Sections D and E already produced. `references` carries
# both within-cohort anchors and `unadapted` carries the transfer one, all three fitted under
# the loaded settings — the same computation this loop would repeat.
_tuned = pd.concat([references, unadapted], ignore_index=True)
_SCREEN_COLS = (["family", "arm", "hyperparameter_source", "seed", "threshold", "eval_config"]
                + list(evaluation.SUMMARY_METRICS) + ["prevalence"])
_tuned = (_tuned.rename(columns={"regime": "eval_config"}).assign(arm="tuned"))

# Both arms on the same columns. The confusion counts and denominators behind the rate
# metrics are not carried: this frame is displayed, and an exact MCS count does not need to be.
configuration_screening = pd.concat(
    [pd.DataFrame(_rows)[_SCREEN_COLS], _tuned[_SCREEN_COLS]], ignore_index=True)
print(f"{len(configuration_screening):,} rows | "
      f"{configuration_screening['family'].nunique()} families x "
      f"{configuration_screening['eval_config'].nunique()} evaluations x "
      f"{configuration_screening['arm'].nunique()} arms x "
      f"{configuration_screening['seed'].nunique()} seeds")
print("  tuned rows are Sections D and E's; only the untuned arm was fitted here")

### Result — inside each cohort

Two numbers that never cross a border: each cohort's own model on its own held-out records.

In [ ]:
print("trained and evaluated inside one cohort — mean AUC over twenty seeds")
display(configuration_screening[configuration_screening["eval_config"] != "unadapted"]
        .pivot_table(index=["family", "threshold"], columns=["eval_config", "arm"], values="auc")
        .rename(columns=regime_names.SHORT_DISPLAY, level="eval_config")
        .reindex(regime_names.FAMILIES, level="family").round(4))

### Result — the arms after transfer

The same MCS models applied to YRBS unchanged, at both arms. Read the `difference` column as an
evaluation of the loaded selection, not as its justification.

In [ ]:
screening_unadapted_auc = (
    configuration_screening[configuration_screening["eval_config"] == "unadapted"]
    .pivot_table(index=["family", "threshold"], columns="arm", values="auc")
    .reindex(regime_names.FAMILIES, level="family"))

screening_unadapted_auc["difference"] = (screening_unadapted_auc["tuned"]
                                         - screening_unadapted_auc["untuned"])

print("unadapted transfer — mean AUC over twenty seeds")
display(screening_unadapted_auc.round(4))

# A count, not a test: these are means over twenty seeds and no pairing has been done
# here. The count and the comparison with the earlier run are both internal.
if not PUBLIC_NOTEBOOK:
    _d = screening_unadapted_auc["tuned"] - screening_unadapted_auc["untuned"]
    print(f"\ntuned above untuned in {int((_d > 0).sum())} of {len(_d)} "
          f"(family, threshold) cells; median difference {_d.median():+.4f}")
    print("The manuscript (§IV, line 252) reports that tuning 'changed MCS performance")
    print("little but improved unadapted transfer at both thresholds'. That is an")
    print("observation from the run the manuscript was written from; the line above is")
    print("this run's.")

### The settings each family runs under

**One tracked specification holds the configuration.** `spec/local_model_settings.csv` carries
both cohorts' 54 fixed configurations and two provenance identifiers — `protocol_id` for the
procedure, `settings_digest` for the configurations themselves — and nothing else. The
cross-validated values behind each choice stay in the private records under the working root and
are never promoted.

How it was reached is Section D, and it is reproducible: the same procedure re-run on the same
development partitions reaches the same 54 cells. The cell below refuses to continue unless the
loaded specification is complete and self-consistent over this run's thresholds and families.

In [ ]:
# The tracked specification, validated through the live loader rather than described.
# One combined file carries both cohorts' fixed configurations; what is checked here is
# that this run is fitted under a complete, self-consistent copy of it.
#
# NO CROSS-VALIDATED VALUE IS DISPLAYED. The seed-level AUCs behind each choice, their
# mean and their standard deviation live in the private selection records under the
# working root; the MCS ones are MCS-derived and need separate disclosure review. What a
# reader needs here is which configuration each model was fitted under.
_spec_loaded = models.load_local_settings()
if set(_spec_loaded) != {"mcs", "yrbs"}:
    raise ValueError(f"the specification carries cohort key(s) {sorted(_spec_loaded)}; "
                     f"the declared cohorts are ['mcs', 'yrbs']")

expected_cells = len(THRESHOLDS) * len(FAMILIES)
for _cohort, _mapping in _spec_loaded.items():
    if len(_mapping) != expected_cells:
        raise ValueError(
            f"the {_cohort} specification carries {len(_mapping)} cells, expected "
            f"{expected_cells} ({len(THRESHOLDS)} thresholds x {len(FAMILIES)} families)")
    _absent = [(t, f) for t in THRESHOLDS for f in FAMILIES if (t, f) not in _mapping]
    if _absent:
        raise ValueError(f"the {_cohort} specification is missing {_absent}")
    if any(len(_key) != 2 for _key in _mapping):
        raise ValueError(
            f"a {_cohort} key is not (threshold, family). A seed in the key would mean "
            f"the twenty splits did not share one configuration.")

# The mappings this notebook is actually fitting under are the ones just validated.
if _spec_loaded["mcs"] != dict(MCS_SETTINGS) or _spec_loaded["yrbs"] != dict(YRBS_SETTINGS):
    raise ValueError("the loaded specification differs from the mappings in use")

# The declared metadata and schema, read as text. One procedure and one promotion behind
# both halves: two different selections run under the same procedure share a protocol_id
# and differ in settings_digest, so a matching protocol_id alone is not evidence that two
# sets of results are comparable.
_spec = pd.read_csv(config.LOCAL_MODEL_SETTINGS, dtype=str, keep_default_na=False)
if list(_spec.columns) != list(models.SETTINGS_COLUMNS):
    raise ValueError(f"the tracked specification carries columns "
                     f"{list(_spec.columns)}; expected {list(models.SETTINGS_COLUMNS)}")
_protocols = sorted(set(_spec["protocol_id"]))
_digests = sorted(set(_spec["settings_digest"]))
if _protocols != [models.protocol_id()]:
    raise ValueError(f"the specification declares protocol_id {_protocols}; the live "
                     f"procedure is {models.protocol_id()!r}")
if len(_digests) != 1:
    raise ValueError(f"the specification carries {len(_digests)} settings_digest "
                     f"values; one file describes one promotion")

print(f"{len(_spec_loaded['mcs'])} MCS and {len(_spec_loaded['yrbs'])} YRBS "
      f"configurations, {len(_spec)} rows in spec/local_model_settings.csv")
print(f"  protocol_id      {_protocols[0]}")
print(f"  settings_digest  {_digests[0]}")
print("  each cohort selected inside its own outer-training partitions on development "
      "seeds 0, 1 and 2;")
print("  no outer test partition of either cohort entered either selection")

# The configurations themselves, both cohorts side by side. No score column.
display(_spec[_spec.family.isin(FAMILIES)]
        .pivot(index=["threshold", "family"], columns="cohort", values="settings"))

### The configuration, as a table

Nothing is decided here; this is the record of what was decided and where it came from.

**Read the `status` column first.** *loaded* means an earlier development stage chose it and
this notebook reads the choice. *declared* means it is a constant in `src/` or in the scope
cell above. *recomputed* means this run derives it. *frozen* means it lives under `outputs/`
from an earlier run and nothing here rewrites it.

Where a rationale is not recoverable from this repository or from the draft, the table says
**rationale to document** rather than supplying one. Section N carries those as open questions.

In [ ]:
DECISIONS = [
    # decision, value, source, status, rationale
    ("outcome", "count of 5 shared ACE pillars, strict composition",
     "outcomes.make_outcome", "declared",
     "one definition; strict=False turns 'not asked' into 'not exposed'"),
    ("thresholds", ">=1, >=2, >=3",
     "notebook scope cell", "declared",
     "the analysis reports the lower, primary and higher sensitivity cuts; >=2 is primary "
     "and no >=4 model is fitted"),
    ("families", f"{len(regime_names.FAMILIES)}: " + ", ".join(regime_names.FAMILIES),
     "regime_names.FAMILY_CLASS", "declared",
     "spans the three model classes the reporting partitions on (linear, bagged, boosted)"),
    ("seed protocol", "20 seeds; stratified 75/25 per cohort; model seed fixed at 42",
     "data.build_splits, models.RANDOM_STATE", "declared",
     "shared splits make every comparison paired within a seed (§IV)"),
    ("primary metrics", "AUC and PR-AUC; Brier, ECE and calibration slope alongside",
     "scores.METRIC_GROUPS", "declared",
     "discrimination and calibration are reported together because they need not move together"),
    ("PR-AUC null", "cohort prevalence at each cut, carried on every row",
     "evaluation.metrics", "recomputed",
     "a PR-AUC without its no-skill null is unreadable across cohorts"),
    ("hyperparameters",
     "three-seed consensus inside each cohort: 5-fold stratified inner CV AUC on "
     "development seeds 0, 1 and 2, mean of the three seed-level means, fixed "
     "across the 20 evaluation seeds",
     "Section D, promoted to spec/local_model_settings.csv", "loaded",
     "re-selecting per run would make the across-seed spread a search artefact; "
     "neither cohort's selection read the other's data or any outer test partition"),
    ("arm", "tuned",
     "transfer.ARM", "declared",
     "the reported arm; the untuned arm is evaluated in Section F and not carried further"),
    (
        "predictor encoding",
        f"{len(data.FEATURE_COLUMNS)} harmonised predictors -> "
        f"{len(data.MODEL_FEATURE_COLUMNS)} model columns; the one nominal "
        f"predictor enters as one indicator per non-reference level",
        "data.model_features, applied in Section B before any split",
        "declared",
        "its codes are level names rather than a scale, so a single numeric "
        "column would impose an order on the levels",
    ),
    ("preprocessing", "cohort standardisation, each frame against itself",
     "data.standardise_cohort, fitted inside data.build_splits", "declared",
     "counted as a label-free adaptation (§V-B); fitted per split in Section B, not "
     "upstream — see Section C for what it means for the 'unadapted' baseline"),
    ("label budget k", f"k = {transfer.K}, from a pool draw disjoint from the test set",
     "transfer.K, SplitBundle.k_slice", "declared",
     "k = 500 is a pragmatic common comparison point, not a general sufficiency threshold; "
     "the separate nested budget curve evaluates k = 50 to 2000"),
    ("post-adaptation recalibration",
     "5-fold out-of-fold prediction of the k=500 slice, 3 when the minority class supports "
     "three but not five, non-estimable below three; logistic mapping fitted from those "
     "out-of-fold predictions alone; final adaptation on all k=500; evaluation set untouched "
     "until final scoring",
     "transfer.crossfit_logistic_recal_scores; §V-F", "recomputed",
     "the manuscript describes this procedure and the pipeline did not reproduce it. NOT "
     "platt_frozen or isotonic_recal, which recalibrate the frozen source in-sample, and NOT "
     "the Section K sweep, which is a single update/calibrate split at declared ratios"),
    ("budget curve", "k = 50..2000, draws nested so each budget extends the smaller",
     "SplitBundle.nested_draw", "declared",
     "nesting makes comparisons across k paired within a seed"),
    ("reporting groups", f"label-free ({len(scores.LABEL_FREE) - 1} compared), "
     f"post-training ({len(scores.POST_TRAINING)}), label-using ({len(scores.LABEL_USING)})",
     "scores.LABEL_FREE / POST_TRAINING / LABEL_USING", "declared",
     "§V-D separates probability adjustment from adaptation. These organise the reporting; "
     "they are not multiplicity families, because no test is reported"),
    ("paired comparison", "mean difference across the twenty shared splits, with its spread",
     "scores.method_gap", "recomputed",
     "the splits overlap, so the per-seed differences are not independent and no inferential "
     "test is reported — see Section N"),
    ("robustness scope", "L1_LR, XGB, CatBoost",
     "transfer.leave_one_pillar_out, outcome_variant_battery", "declared",
     "a restricted sensitivity analysis over the declared families; its results are not "
     "treated as covering all nine families"),
    ("published tables", "everything under outputs/",
     "outputs/FROZEN.md", "frozen",
     "the freeze the draft was written from; nothing in this notebook republishes over it"),
]

decisions = pd.DataFrame(DECISIONS,
                         columns=["decision", "value", "source", "status", "rationale"])
display(decisions.set_index("decision"))

---
# G — Label-free experiments

**Question.** If the importing authority has questionnaire answers but no recorded outcomes,
how much of the gap in Section E can it close?

Four procedures, and what each assumes. **Quantile mapping** aligns each MCS feature onto the
YRBS pool's whole marginal and trains on the mapped frame — it assumes the discrepancy is in the
marginals. **Importance weighting** reweights the MCS training set by density ratios from a
domain classifier — it assumes covariate shift, which is the assumption cross-jurisdictional
transfer puts under strain. **Pseudo-labelling** runs one round: the source model scores the
unlabelled target pool and its most confident records enter training with their predicted
labels, selected by decile; confident is not correct. **Thresholded self-training** runs three
rounds over that same unlabelled pool, selecting by fixed confidence thresholds rather than by
decile and rescoring the pool after each refit, so the set it trains on can grow or change
between rounds.

**None of the four observes a target training outcome, and none observes the target outcome
rate.** Every label any of them trains on is either an MCS outcome or one the source model
predicted; the YRBS training outcomes are never read.

Cohort standardisation is a fifth label-free adaptation, fitted inside the splits in Section B,
so all four run on top of it: rung 3 of Section C's ladder measured against rung 2. **Prior
correction is not here** — it adjusts probabilities after training rather than adapting the
model, and §V-D evaluates adjustment separately from adaptation, so it is read in Section I.

A plausible expectation to examine: all four alter model fitting using target covariates or
source-model predictions, but none observes target outcomes or the true target outcome rate.
They may therefore move discrimination without repairing calibration. That is a reading of the
methods, not a recorded prediction.

**How it is read.** Each procedure is paired against the baseline within (family, arm,
threshold, seed), and the per-seed differences are summarised by their mean across the twenty
shared splits with the across-split spread beside it. No test is reported: the splits overlap,
so the differences are not independent observations. A small mean difference is reported as a
small mean difference, not as an absence.

**Status: primary analysis.**

### Quantile mapping

Fits its own model on the mapped frame; no source model is used.

In [ ]:
quantile_mapping = run_procedure("quantile_map", transfer.quantile_map_scores)
display(compare_with_references(quantile_mapping))

### Importance weighting

Fits its own weighted model. The canonical source model is used only on the fallback path, for a family whose estimator takes no `sample_weight`.

In [ ]:
importance_weighting = run_procedure("importance_weight", transfer.importance_weight_scores,
                                     sources=source_models)
display(compare_with_references(importance_weighting))

### Pseudo-labelling

The canonical source model labels the top and bottom decile of the unlabelled pool; a fresh model is then trained on MCS plus those records.

In [ ]:
pseudo_labelling = run_procedure("pseudo_label", transfer.pseudo_label_scores,
                                 sources=source_models)
display(compare_with_references(pseudo_labelling))

### Thresholded self-training

Three rounds of confidence-threshold self-training seeded by the canonical source model. It reads the YRBS training covariates and no YRBS training outcome: each round scores the whole unlabelled pool, keeps every record above 0.8 or below 0.2 as a pseudo-label, and refits on MCS plus those records. Distinct from the pseudo-labelling above, which is one round and selects by decile rather than by a fixed threshold.

In [ ]:
threshold_self_training = run_procedure(
    "pseudo_label_thresh", transfer.pseudo_label_thresh_scores, sources=source_models)
display(compare_with_references(threshold_self_training))

### The label-free comparison

Only the frames this comparison needs.

In [ ]:
label_free_results = pd.concat(
    [unadapted, quantile_mapping, importance_weighting, pseudo_labelling,
     threshold_self_training], ignore_index=True)

LABEL_FREE = ["unadapted", "quantile_map", "importance_weight", "pseudo_label",
              "pseudo_label_thresh"]
if set(LABEL_FREE) != set(scores.LABEL_FREE):
    raise ValueError("this list and the reporting group in src/ disagree")
print("Performance by family across twenty predefined splits")

label_free_display = pd.concat(
    {
        "AUC mean": method_grid(
            label_free_results, LABEL_FREE, "auc_mean"
        ),
        "AUC SD": method_grid(
            label_free_results, LABEL_FREE, "auc_sd"
        ),
        "PR-AUC mean": method_grid(
            label_free_results, LABEL_FREE, "prauc_mean"
        ),
        "PR-AUC SD": method_grid(
            label_free_results, LABEL_FREE, "prauc_sd"
        ),
    },
    axis=1,
)

display(label_free_display)

In [ ]:
print("Calibration after the label-free procedures")

display(
    calibration_grid(
        label_free_results,
        LABEL_FREE,
    )
)

In [ ]:
label_free_gaps = {k: scores.method_gap(label_free_results, k, baseline="unadapted")
                   for k in LABEL_FREE if k != "unadapted"}

print("mean change against the cohort-standardised baseline, over twenty shared splits")
# The paired-seed count is protocol rather than result, and averaging it would put the
# seed design into the table as though it were a metric. Dropped before the mean.
display(pd.DataFrame({regime_names.paper_label(k): g.drop(columns="n_paired")
                      .mean(numeric_only=True)
                      for k, g in label_free_gaps.items()}).round(5))

# The per-seed differences themselves, summarised descriptively. Paired within
# (family, arm, threshold, seed); the twenty splits overlap, so these describe how the
# difference behaves across the splits used and are not a test.
label_free_deltas = pd.concat(
    [scores.method_gap(label_free_results, k, baseline="unadapted", aggregate=False)
     for k in LABEL_FREE if k != "unadapted"], ignore_index=True)

_label_free_deltas = (label_free_deltas
                      .assign(display=label_free_deltas["regime"]
                              .map(regime_names.SHORT_DISPLAY))
                      .groupby(["threshold", "display"])["auc"])

print("\nAUC difference from the baseline: mean and spread across the twenty splits")
display(_label_free_deltas.agg(mean="mean", split_sd="std").round(4))

# The range across the splits and the count of cells behind it are diagnostics of the
# draw rather than reported quantities, so they stay internal.
if not PUBLIC_NOTEBOOK:
    display(_label_free_deltas.agg(cells="size", minimum="min", maximum="max")
            .round(4))

if not PUBLIC_NOTEBOOK:
    _above = (pd.DataFrame({k: g.set_index(["family", "threshold"])["d_auc_mean"]
                            for k, g in label_free_gaps.items()}) > 0).sum()
    print("\n(family, threshold) cells where the mean difference is above zero:")
    for k, n in _above.items():
        print(f"  {regime_names.SHORT_DISPLAY[k]:<18} {int(n)} of "
              f"{len(label_free_gaps[k])}")

This section can only speak about procedures that see target *features*. Whether target
*outcomes* buy something different in kind is Section H; whether the calibration half can be
addressed without touching the ranking is Section I.

---
# H — Label-using experiments at k = 500

**Question.** Where outcomes can be obtained for some of the authority's own adolescents, what does spending them buy?

Every procedure here uses 500 labelled YRBS records from `SplitBundle.k_slice(500)`, drawn from the training partition so it cannot intersect the test set. Whether 500 is the right number is examined in Section J; its operational realism remains a limitation.

**Grouped by mechanism.** These procedures make qualitatively different changes. An isotonic mapping and a coefficient-sign constraint cannot be ordered meaningfully by the number of parameters they change, and the procedures do not all apply to the same model families. Each is compared with unadapted transfer rather than ranked against the other procedures.

| group | what changes |
|---|---|
| **recalibration** | the relationship between score and probability; ranking is preserved by logistic recalibration, while isotonic recalibration can introduce ties |
| **constrained updating** | part of the source model remains fixed while another component is estimated using target outcomes |
| **ensembling** | the source model is combined with a model fitted using target outcomes |
| **target-only and full revision** | the source model is discarded or all model parameters are re-estimated |

**Two caveats on the names.** Eleven procedures are computed and reported. `leaf_refresh_global` is a one-parameter intercept offset rather than a complete leaf refresh. `feature_set` uses the source-selected active predictors rather than the full set of 30 harmonised predictors represented by 31 model columns.

**How it is read.** As in Section G, results are summarised using the mean paired difference across the twenty shared splits and its spread. The eleven procedures are grouped for reporting; no multiplicity-adjusted significance analysis is presented in this notebook.

**Status: primary analysis.**

## Recalibration

Both refit the link between the source model's score and a probability and leave the model alone. Read in Section I.

### Platt scaling on the frozen source

Intercept and slope on the canonical source model's logit — strictly increasing, so the ranking is untouched.

In [ ]:
platt_recalibration = run_procedure("platt_frozen", transfer.platt_frozen_scores,
                                    sources=source_models)
display(compare_with_references(platt_recalibration))

### Isotonic recalibration

A monotone non-decreasing link refitted on the source scores. Its flat segments tie scores, so AUC can fall and can never rise.

In [ ]:
isotonic_recalibration = run_procedure("isotonic_recal", transfer.isotonic_recal_scores,
                                       sources=source_models)
display(compare_with_references(isotonic_recalibration))

## Constrained updating

Part of the source model is held fixed and the rest re-estimated. What is held fixed differs: an intercept offset holds everything, a feature set holds only which columns enter. The three coefficient-transfer procedures take their coefficients from the canonical **L1_LR** source, whatever family is under test.

### Global intercept offset (trees only)

One degree of freedom on the source logit. **Not** the leaf refresh of the write-up — that is Section L.

In [ ]:
intercept_offset = run_procedure(
    "leaf_refresh_global", transfer.leaf_refresh_global_scores,
    families=[f for f in FAMILIES if f not in models.LR_FAMS], sources=source_models)
display(compare_with_references(intercept_offset))

### Frozen source coefficients (LR only)

The L1-LR source coefficients are frozen and only the intercept is re-estimated.

In [ ]:
coefficient_freezing = run_procedure(
    "coef_freeze_intercept", transfer.coef_freeze_intercept_scores,
    families=models.LR_FAMS, sources=source_models,
    source_arg="l1_source", source_family="L1_LR")
display(compare_with_references(coefficient_freezing))

### Sign and support constraints (LR only)

Keeps the source's active set and coefficient signs, re-estimating magnitudes on the target outcomes.

In [ ]:
sign_support = run_procedure(
    "sign_support", transfer.sign_support_scores,
    families=models.LR_FAMS, sources=source_models,
    source_arg="l1_source", source_family="L1_LR")
display(compare_with_references(sign_support))

### Source feature set (LR only)

Keeps only the features the L1 source selected and retrains freely on them. **Not** the 31-feature harmonised schema.

In [ ]:
source_feature_set = run_procedure(
    "feature_set", transfer.feature_set_scores,
    families=models.LR_FAMS, sources=source_models,
    source_arg="l1_source", source_family="L1_LR")
display(compare_with_references(source_feature_set))

## Ensembling and self-training

The source model is combined with a target-side model, or used to label target records for one. Both ensembles fit their own target model; the source side is a canonical model, used read-only.

### Ensemble with a target model

A convex blend of the canonical source model with a target-trained model of the same family.

In [ ]:
ensemble_same_family = run_procedure(
    "ensemble_same_family", transfer.ensemble_same_family_scores, sources=source_models)
display(compare_with_references(ensemble_same_family))

### Ensemble on a CatBoost source

The same blend with the canonical **CatBoost** source on the source side. Excluded for CatBoost itself, where it would be the same-family ensemble.

In [ ]:
ensemble_catboost_source = run_procedure(
    "ensemble_catboost_source", transfer.ensemble_catboost_source_scores,
    families=[f for f in FAMILIES if f != "CatBoost"], sources=source_models,
    source_arg="cb_source", source_family="CatBoost")
display(compare_with_references(ensemble_catboost_source))

## Target-only and full revision

The source model is discarded, or every parameter in it re-estimated.

### Target-only

Trains on the 500 target outcomes alone; the source model plays no part in the method. It is passed one anyway, used only if the drawn slice carries a single class and the procedure has to fall back to unadapted transfer. A different quantity from Section E's target-trained reference, which uses the whole pool.

In [ ]:
target_only, target_only_scores = run_procedure(
    "target_only", transfer.target_only_scores, sources=source_models, keep_scores=True)
display(compare_with_references(target_only))

# FOCAL SCOPE. The procedure runs on all nine families and the metric frame keeps all nine.
# Only L1_LR's predictions have a downstream reader — it is the target-only pipeline the draft
# carries through §VI-C's fixed-capacity results and Table V's subgroup gains — so the other
# eight families' per-person scores are dropped here rather than persisted unread.
target_only_scores = target_only_scores[target_only_scores["family"] == "L1_LR"]
if not PUBLIC_NOTEBOOK:
    print(f"\nfocal scores retained: {len(target_only_scores):,} rows, L1_LR only")

### Full revision

Every parameter re-estimated: a warm start from the canonical source model for the three families that support it, otherwise a weighted refit on MCS plus the target outcomes. The manuscript calls this *full revision*; the pipeline key is `fine_tune`.

In [ ]:
full_revision, full_revision_scores = run_procedure(
    "fine_tune", transfer.fine_tune_scores, sources=source_models, keep_scores=True)
display(compare_with_references(full_revision))

# FOCAL SCOPE, as above. Random forest, HistGB and CatBoost are the three full-revision
# pipelines the draft reports; the other six families' per-person scores are dropped.
FOCAL_FINE_TUNE = ["RF", "HistGB", "CatBoost"]
full_revision_scores = full_revision_scores[
    full_revision_scores["family"].isin(FOCAL_FINE_TUNE)]
if not PUBLIC_NOTEBOOK:
    print(f"\nfocal scores retained: {len(full_revision_scores):,} rows, "
          f"{', '.join(FOCAL_FINE_TUNE)}")

### Raw-feature L1 head

An L1 logistic head on **raw** target features and the 500 outcomes, ignoring the source cohort
and the cohort standardisation alike. Family- and arm-independent by construction, so it is
fitted once per (threshold, seed) and the same scores recorded for every family. Its unit of
work is a seed where `run_procedure`'s is a family, so it keeps its own loop, with the same
per-cell checks. Its scores are not among those kept.

In [ ]:
_name, rows, seen, started = "raw_l1_head", [], set(), time.time()
for t in THRESHOLDS:
    for seed in SEEDS:
        S = splits[(seed, t)]
        s_raw = S.k_slice(transfer.K, lineage="raw")
        head = transfer.raw_l1_head(s_raw[S.feat_cols], s_raw["y"], S["Xy_te"], seed)
        sc = transfer.raw_l1_head_scores(S, rawl1=head)      # family-independent by construction
        if set(sc) != {_name}:
            raise RuntimeError(f"{_name} at {(t, seed)} emitted {sorted(sc)}")
        for fam in FAMILIES:
            _, hsrc = models.select_params(fam, "tuned", t, MCS_SETTINGS)
            r, _f, mismatched = transfer.metric_rows(
                sc, S, family=fam, arm="tuned", threshold=t, seed=seed, source=hsrc,
                keep_scores=False)
            if len(r) != 1:
                raise RuntimeError(f"{_name} at {(fam, t, seed)}: {len(r)} rows for 1 regime")
            if mismatched:
                raise RuntimeError(
                    f"{_name} at {(fam, t, seed)}: score rows do not match the YRBS test "
                    f"index ({mismatched}).")
            rows += r; seen.add((fam, t, seed))

if seen != {(f, t, s) for f in FAMILIES for t in THRESHOLDS for s in SEEDS}:
    raise ValueError(f"{_name}: the loop did not cover every family, threshold and seed")
raw_l1_head = pd.DataFrame(rows)
if not PUBLIC_NOTEBOOK:
    print(f"{_name}: {len(raw_l1_head):,} rows | {time.time() - started:.0f}s")
display(compare_with_references(raw_l1_head))

### The label-using comparison

The eleven frames, concatenated with the baseline.

In [ ]:
MECHANISM = {
    "recalibration": {"platt_frozen": platt_recalibration,
                      "isotonic_recal": isotonic_recalibration},
    "constrained updating": {"leaf_refresh_global": intercept_offset,
                             "coef_freeze_intercept": coefficient_freezing,
                             "sign_support": sign_support,
                             "feature_set": source_feature_set},
    "ensembling": {"ensemble_same_family": ensemble_same_family,
                   "ensemble_catboost_source": ensemble_catboost_source},
    "target-only and full revision": {"target_only": target_only, "fine_tune": full_revision,
                                      "raw_l1_head": raw_l1_head},
}
GROUP_OF = {k: g for g, d in MECHANISM.items() for k in d}
if set(GROUP_OF) != set(scores.LABEL_USING):
    raise ValueError("the mechanism groups here and the reporting group in src/ disagree")

label_using_results = pd.concat(
    [unadapted] + [f for d in MECHANISM.values() for f in d.values()], ignore_index=True)
for group, d in MECHANISM.items():
    print(f"  {group:<30} {', '.join(regime_names.SHORT_DISPLAY[k] for k in d)}")

In [ ]:
print(
    f"Discrimination at k = 500 across twenty predefined splits, "
    f"{HEADLINE_THRESHOLD}"
)

label_using_discrimination = pd.concat(
    {
        "AUC mean": method_grid(
            label_using_results,
            list(GROUP_OF),
            "auc_mean",
        ),
        "AUC SD": method_grid(
            label_using_results,
            list(GROUP_OF),
            "auc_sd",
        ),
        "PR-AUC mean": method_grid(
            label_using_results,
            list(GROUP_OF),
            "prauc_mean",
        ),
        "PR-AUC SD": method_grid(
            label_using_results,
            list(GROUP_OF),
            "prauc_sd",
        ),
    },
    axis=1,
)

display(label_using_discrimination)

In [ ]:
print("Calibration after the label-using procedures")

for mechanism, procedures in MECHANISM.items():
    # These are reported with BBSE in the dedicated probability-adjustment table.
    if mechanism == "recalibration":
        continue

    procedure_keys = ["unadapted"] + list(procedures)

    print(f"\n{mechanism}")
    display(
        calibration_grid(
            label_using_results,
            procedure_keys,
        )
    )

The ECE grid is the half of the table the two recalibration procedures exist for: neither can raise AUC, so a discrimination-only reading would conclude they did nothing.

In [ ]:
label_using_gaps = pd.concat(
    [scores.method_gap(label_using_results, k, baseline="unadapted")
       .assign(display=regime_names.SHORT_DISPLAY[k], mechanism=GROUP_OF[k])
     for k in GROUP_OF], ignore_index=True).rename(columns={"d_auc_mean": "delta_auc"})

# Descriptive, by mechanism: how the AUC difference from the baseline behaves across the
# twenty shared splits. Paired within (family, arm, threshold, seed); no test is reported.
label_using_deltas = pd.concat(
    [scores.method_gap(label_using_results, k, baseline="unadapted", aggregate=False)
       .assign(display=regime_names.SHORT_DISPLAY[k], mechanism=GROUP_OF[k])
     for k in GROUP_OF], ignore_index=True)

_label_using_deltas = label_using_deltas.groupby(["mechanism", "display"])["auc"]

print("AUC difference from the baseline, over twenty shared splits")
display(_label_using_deltas.agg(mean="mean", split_sd="std").round(4))

# The range and the cell count behind the same differences, internal for the same
# reason: they describe the draw rather than the result.
if not PUBLIC_NOTEBOOK:
    display(_label_using_deltas.agg(cells="size", minimum="min", maximum="max")
            .round(4))

In [ ]:
print("change against the baseline, by mechanism and procedure")
# The median alone. The per-procedure tables above already show the individual
# results, so a range here would describe the spread across families rather than add
# a further result.
display(label_using_gaps.groupby(["mechanism", "display"])["delta_auc"]
        .agg(["median"]).round(4))
if not PUBLIC_NOTEBOOK:
    display(label_using_gaps.groupby(["mechanism", "display"])["delta_auc"]
            .agg(["min", "max"]).round(4))
print("\nmedian change by mechanism group")
display(label_using_gaps.groupby("mechanism")[["delta_auc", "d_ece_mean"]].median().round(4))

In [ ]:
panel_using = label_using_gaps[label_using_gaps["threshold"] == ">=2"]
order = [regime_names.SHORT_DISPLAY[k] for k in GROUP_OF
         if regime_names.SHORT_DISPLAY[k] in set(panel_using["display"])][::-1]
adaptation_figure, ax = plt.subplots(figsize=(7.2, max(2.2, 0.34 * len(order) + 1.0)))
for y, disp in enumerate(order):
    grp = panel_using[panel_using["display"] == disp]
    for _, row in grp.iterrows():
        cls = regime_names.FAMILY_CLASS.get(row["family"], "linear")
        ax.scatter(row["delta_auc"], y, s=26, color=regime_names.CLASS_COLOUR[cls],
                   marker=regime_names.CLASS_MARKER[cls], alpha=.85, zorder=3)
    ax.scatter(grp["delta_auc"].median(), y, s=90, facecolor="none",
               edgecolor="#0A2540", lw=1.2, zorder=4)
ax.axvline(0, color="#0A2540", lw=1.0)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel("Δ AUC against the cohort-standardised baseline")
ax.set_title("Label-using procedures against the baseline, >=2  (mechanism order)")
handles = [plt.Line2D([], [], ls="", marker=regime_names.CLASS_MARKER[c],
                      color=regime_names.CLASS_COLOUR[c], label=c)
           for c in ("linear", "bagged", "boosted")]
handles.append(plt.Line2D([], [], ls="", marker="o", mfc="none", mec="#0A2540", label="median"))
ax.legend(handles=handles, frameon=False, ncol=4, loc="lower center",
          bbox_to_anchor=(0.5, -0.30 - 0.02 * len(order)))
plt.show()

Every delta is paired within (family, arm, threshold, seed), so a procedure and the baseline are
read on the same respondents under the same fitted models. The figure is drawn in mechanism
order rather than sorted by effect, and the circled point is the median across families — a
descriptive summary of where that mechanism sits, not a test.

---
# I — Post-training probability adjustment

**Question.** Section E's calibration panel showed the probabilities moving. Can they be
corrected after training, without disturbing which adolescents the model ranks above which?

**Why this is its own section.** These procedures change the numbers a fitted model emits and
leave the model alone. §V-D evaluates adjustment separately from adaptation, which is why they
are read together here rather than among Section G's three.

Prior correction is the only label-free member. `platt_frozen` and `isotonic_recal` spend the
same 500 outcomes as Section H and were run there; this is where they are read. The section
closes with the one adjustment the manuscript describes and the pipeline did not previously
reproduce: cross-fitted logistic recalibration after adaptation.

**Status: primary analysis.**

### Prior correction

Rescales the source model's probabilities for the target outcome rate, estimated without a
single target outcome — from TPR and FPR on the labelled source and the predicted-positive rate
on the unlabelled target pool.

**It spends no target outcome.** The three inputs are: TPR and FPR measured on the *labelled
source* validation slice, and the predicted-positive rate on the *unlabelled* target pool.

$$\hat q = \frac{m - \mathrm{FPR}}{\mathrm{TPR} - \mathrm{FPR}} \qquad w_1 = \hat q/\pi, \quad w_0 = (1-\hat q)/(1-\pi)$$
$$p' = \frac{p\,w_1}{(1-p)\,w_0 + p\,w_1} \qquad \frac{dp'}{dp} = \frac{w_0 w_1}{((1-p)\,w_0 + p\,w_1)^2}$$

Held-out target outcomes are used only afterwards, by the evaluation layer, to score the frozen
corrected predictions. Nothing measured there feeds back into the estimate.

**The condition rank preservation depends on.** The map is strictly increasing **only while both
weights are strictly positive**, i.e. while $0 < \hat q < 1$. Rank preservation is therefore a
property of the *estimate*, not a guarantee of the method: at either boundary the correction is
constant and every score ties. Whether the estimate stays interior on this data is a question,
not an assumption, and the check below is a real test.

In [ ]:
prior_correction = run_procedure("prior_correction", transfer.prior_correction_scores,
                                   expect_regimes=("bbse",), sources=source_models)
display(compare_with_references(prior_correction))

In [ ]:
post_training_results = pd.concat([unadapted, prior_correction], ignore_index=True)

prior_correction_gaps = scores.method_gap(post_training_results, "bbse", baseline="unadapted")

print("mean change against the cohort-standardised baseline, over twenty shared splits")
display(prior_correction_gaps[["family", "threshold", "d_auc_mean", "d_prauc_mean",
                               "d_brier_mean", "d_ece_mean"]]
        .round(5).sort_values("d_auc_mean"))

prior_correction_deltas = scores.method_gap(post_training_results, "bbse",
                                            baseline="unadapted", aggregate=False)
if not PUBLIC_NOTEBOOK:
    print("\nper-split spread of the same differences")
    display(prior_correction_deltas.groupby("threshold")[["auc", "ece", "brier"]]
            .agg(["mean", "std", "min", "max"]).round(4))

### The rank-invariance check, on all three monotone procedures

All three claim to leave the ranking alone, and they claim it for different reasons. Prior
correction is strictly increasing *while its estimate stays interior*. Platt scaling is a
strictly increasing two-parameter map and should hold outright. Isotonic regression is monotone
**non-decreasing**, so its flat segments create ties: its AUC can legitimately fall and can never
rise. That is the specification, as specified, and it is why the three are checked with the same
instrument and read differently.

In [ ]:
# The three monotone procedures, checked on a frame built for this check alone.
monotone_results = pd.concat(
    [unadapted, prior_correction, platt_recalibration, isotonic_recalibration], ignore_index=True)

for reg in ("bbse", "platt_frozen", "isotonic_recal"):
    c = scores.check_rank_invariant(monotone_results, reg)
    print(f"{regime_names.SHORT_DISPLAY[reg]:<22s} checked={c['checked']}  "
          f"holds={c['rank_invariant_holds']}  over {c['cells']} paired cells\n"
          f"{'':22s} largest |delta| on a rank metric  {c['max_abs_rank_delta']:.2e}  "
          f"(tolerance {scores.RANK_TOL:.0e})\n"
          f"{'':22s} largest |delta| on calibration    {c['max_abs_calibration_delta']:.2e}")

print("\nper family: AUC should be 0 where prior correction's map stayed monotone; "
      "calibration should move")
display(prior_correction_gaps[["family", "threshold", "d_auc_mean", "d_prauc_mean",
                               "d_brier_mean", "d_ece_mean"]].round(5).sort_values("d_auc_mean"))

**Interpretation.** $\hat q \in [0,1]$ is a declared boundary rather than a hidden repair. Every
call reports `solver_status` — the eight values are defined in `src/transfer.py` — with the TPR,
FPR, $m$, $\pi$ and both weights behind it. A boundary result returns exact zeros or ones, and a
system that is not identifiable returns NaN rather than the uncorrected scores.
`tests/test_prior_correction.py` pins that behaviour on synthetic data.

In [ ]:
# What the solver actually did, across the grid — the diagnostic behind the invariant above.
display(prior_correction.groupby("solver_status")
          .agg(cells=("seed", "size"),
               valid=("valid_correction", "sum"),
               weights_positive=("weights_strictly_positive", "sum"),
               q_median=("target_prevalence_constrained", "median")).round(4))

### The recalibration ladder

The three adjustment procedures beside the baseline, on discrimination and on calibration
together. This is the table the section exists to produce: a procedure that moves ECE while
holding AUC has bought calibration and nothing else, which is a real purchase and invisible in a
discrimination-only report.

In [ ]:
PROBABILITY_ADJUSTMENTS = [
    "unadapted",
    "bbse",
    "platt_frozen",
    "isotonic_recal",
]

print("Calibration after probability adjustment")

display(
    calibration_grid(
        monotone_results,
        PROBABILITY_ADJUSTMENTS,
    )
)

### Recalibration after adaptation

**Question.** After adapting with 500 target outcomes, does cross-fitted logistic recalibration
improve probability accuracy without changing the underlying evaluation sample?

**Not the two procedures above, and not Section K's sweep.** `platt_frozen` and `isotonic_recal`
recalibrate the **frozen source model**, and both fit their mapping on the same 500 records
in-sample. Section K's sweep adapts on one part of the slice and corrects on another, in a single
split at declared ratios. This is the procedure §V-F describes and neither of those implements:
adaptation first, then a mapping fitted on **out-of-fold** predictions of the same 500 records.

**The procedure**, per focal pipeline, threshold and seed:

1. take the existing k = 500 slice, drawn from the training half;
2. split it into five stratified folds — three when the smaller outcome class cannot support
   five, and neither when it cannot support three, which is reported rather than estimated;
3. for each fold, fit the focal adaptation on the other folds and predict the held-out fold, so
   every one of the 500 records receives exactly one out-of-fold prediction;
4. fit an intercept and slope from those out-of-fold predictions to their outcomes;
5. score the YRBS evaluation frame with the final model adapted on all 500 records;
6. apply the mapping to those scores.

**The evaluation frame enters neither fit.** Folds are drawn inside the k slice; `Xy_te_cs` is
touched once, at the end. Nothing measured on it feeds back.

**Four focal pipelines** — target-only `L1_LR`, and full revision on `RF`, `HistGB` and
`CatBoost`. These are the four the draft carries through §VI-C and §VI-D. The regime name
carries the distinction from the raw procedure, so a recalibrated row is identifiable on
`regime` alone.

**Cost.** Five fold-fits plus one final fit per cell. The final fit repeats the raw procedure's,
which is the price of keeping the procedure self-contained.

**Status: primary analysis.**

In [ ]:
# Two calls, because the focal set is (regime, family) PAIRS rather than a family list:
# target-only is reported for L1_LR and full revision for the three tree families.
target_only_recal, target_only_recal_scores = run_procedure(
    "target_only_logistic_recal", transfer.crossfit_logistic_recal_scores,
    families=["L1_LR"], sources=source_models, keep_scores=True, regime="target_only")

full_revision_recal, full_revision_recal_scores = run_procedure(
    "fine_tune_logistic_recal", transfer.crossfit_logistic_recal_scores,
    families=FOCAL_FINE_TUNE, sources=source_models, keep_scores=True, regime="fine_tune")

recalibrated_focal = pd.concat([target_only_recal, full_revision_recal], ignore_index=True)
recalibrated_focal_scores = pd.concat(
    [target_only_recal_scores, full_revision_recal_scores], ignore_index=True)

# The declared focal set, checked against what ran.
_pairs = {(r.regime.replace("_logistic_recal", ""), r.family)
          for r in recalibrated_focal.itertuples()}
if _pairs != set(transfer.FOCAL_PIPELINES):
    raise ValueError(f"the focal pipelines ran as {sorted(_pairs)}, declared "
                     f"{sorted(transfer.FOCAL_PIPELINES)}")

# Every declared cell was ATTEMPTED. A cell whose k slice cannot support three stratified folds
# has no mapping to fit, and is recorded as non-estimable rather than filled with the
# uncorrected model's scores: its metrics are blank, it writes no person-level rows, and the
# raw result remains available under `target_only` or `fine_tune`.
recal_status_counts = (recalibrated_focal
                       .groupby(["family", "threshold", "recal_status"])
                       .size().rename("cells").reset_index())
print("\nestimability of every attempted recalibration cell")
display(recal_status_counts.pivot_table(index=["family", "threshold"], columns="recal_status",
                                        values="cells", fill_value=0))

RECAL_ESTIMATED = recalibrated_focal["recal_status"] == "estimated"

# Per (regime, family, threshold): how many of the twenty seeds the mapping could be fitted on.
# A CELL IS COMPLETE ONLY IF ALL TWENTY ARE. Anything less and the ordinary across-seed mean
# would be a mean over whichever splits happened to support recalibration, which is a different
# statistic from the one every other row in the battery reports.
recal_estimability = (
    recalibrated_focal.assign(estimated=RECAL_ESTIMATED)
    .groupby(["regime", "family", "threshold"])
    .agg(seeds_attempted=("seed", "nunique"), seeds_estimated=("estimated", "sum"))
    .reset_index()
    .assign(seeds_expected=len(SEEDS)))
recal_estimability["seeds_non_estimable"] = (
    recal_estimability["seeds_attempted"] - recal_estimability["seeds_estimated"])
recal_estimability["recalibration_estimability_status"] = np.where(
    recal_estimability["seeds_estimated"] == recal_estimability["seeds_expected"],
    "complete", "incomplete")
def non_estimable_reasons(statuses):
    """One cell's reasons for not being estimated, as `reason=count` pairs."""
    return "; ".join(f"{reason}={n}" for reason, n in statuses.value_counts().items())


_reasons = (recalibrated_focal[~RECAL_ESTIMATED]
            .groupby(["regime", "family", "threshold"])["recal_status"]
            .agg(non_estimable_reasons)
            .rename("non_estimable_reasons").reset_index())
recal_estimability = recal_estimability.merge(
    _reasons, on=["regime", "family", "threshold"], how="left")
recal_estimability["non_estimable_reasons"] = (
    recal_estimability["non_estimable_reasons"].fillna(""))

print("\nestimability per (regime, family, threshold)")
display(recal_estimability.set_index(["regime", "family", "threshold"]))
n_attempted, n_estimated = len(recalibrated_focal), int(RECAL_ESTIMATED.sum())
print(f"\n{n_attempted} cells attempted | {n_estimated} estimated | "
      f"{n_attempted - n_estimated} non-estimable")
if n_attempted != n_estimated:
    print("  non-estimable cells, by reason:")
    for reason, n in (recalibrated_focal.loc[~RECAL_ESTIMATED, "recal_status"]
                      .value_counts().items()):
        print(f"    {reason}: {n}")
print("\nfolds used where the mapping was estimable")
display(recalibrated_focal[RECAL_ESTIMATED].groupby(["family", "threshold"])["recal_folds"]
        .value_counts().rename("cells").reset_index()
        .pivot_table(index=["family", "threshold"], columns="recal_folds",
                     values="cells", fill_value=0))

### What the correction bought

Raw against recalibrated, on the two metrics the correction is meant to move and the one whose
behaviour depends on the fitted mapping.

**The mapping is not constrained to be increasing.** It is an unpenalised two-parameter logistic
fitted on out-of-fold predictions, and its coefficient can come out **positive** — the ranking is
preserved and AUC should be unchanged apart from floating-point and tie effects — **zero**, where
every prediction becomes the same value, or **negative**, where the ranking reverses. §V-F
specifies "an intercept and slope relating model scores to outcomes" and defines no
increasing-only recalibrator, so the fitted sign is **reported, not imposed**. The cell below
counts the signs across every family x threshold x seed cell rather than averaging them first: an
average slope can be positive while individual cells are not.

In [ ]:
raw_focal = pd.concat(
    [target_only[target_only["family"] == "L1_LR"],
     full_revision[full_revision["family"].isin(FOCAL_FINE_TUNE)]], ignore_index=True)

# THE COMPARISON IS SHOWN ONLY WHERE ALL TWENTY PAIRED SEEDS EXIST. Comparing a twenty-seed raw
# mean with a recalibrated mean taken over whichever seeds were estimable would put two
# different statistics in adjacent columns.
_complete = recal_estimability[
    recal_estimability["recalibration_estimability_status"] == "complete"]
_complete_cells = set(map(tuple, _complete[["family", "threshold"]].to_numpy()))

_est = recalibrated_focal[RECAL_ESTIMATED]
_in_complete = _est.set_index(["family", "threshold"]).index.isin(_complete_cells)

# Seed accounting, and internal for that reason: what is public below is the comparison.
if not PUBLIC_NOTEBOOK:
    print(f"complete paired cells: {len(_complete_cells)} of "
          f"{len(recal_estimability)}; raw seeds expected {len(SEEDS)} in every cell")
    display(recal_estimability[["regime", "family", "threshold", "seeds_expected",
                                "seeds_attempted", "seeds_estimated",
                                "seeds_non_estimable",
                                "recalibration_estimability_status",
                                "non_estimable_reasons"]]
            .set_index(["regime", "family", "threshold"]))

if _complete_cells:
    comparison_metrics = [
        "auc",
        "prauc",
        "cal_intercept",
        "cal_slope",
        "brier",
        "ece",
    ]

    raw_complete = raw_focal[
        raw_focal.set_index(["family", "threshold"]).index.isin(
            _complete_cells
        )
    ]

    before = (
        raw_complete
        .groupby(["family", "threshold"])[comparison_metrics]
        .agg(["mean", "std"])
    )

    after = (
        _est[_in_complete]
        .groupby(["family", "threshold"])[comparison_metrics]
        .agg(["mean", "std"])
    )

    recalibration_comparison = pd.concat(
        {
            "before recalibration": before,
            "after cross-fitted recalibration": after,
        },
        axis=1,
    )

    print(
        f"\nRaw and cross-fitted recalibrated results over all "
        f"{len(SEEDS)} paired splits — complete cells only"
    )
    display(recalibration_comparison.round(4))
else:
    print(
        "\nNo cell has all twenty seeds estimable, so no "
        "raw-versus-recalibrated comparison is shown."
    )

# EVERYTHING BELOW IS A DIAGNOSTIC of the fitted mapping — its sign, its clipping, its
# score range and its per-cell parameters — rather than a performance result, so the
# public copy carries the comparison above and none of it.
if not PUBLIC_NOTEBOOK:
    _incomplete = recal_estimability[
        recal_estimability["recalibration_estimability_status"] == "incomplete"]
    if len(_incomplete):
        print(f"\n{len(_incomplete)} incomplete cell(s): reported as incomplete rather "
              f"than compared on their surviving seeds.")

    # The fitted sign, counted per cell against a declared tolerance. Never an average.
    _dir = np.select(
        [_est["recal_slope"] > transfer.RECAL_SLOPE_TOL,
         _est["recal_slope"] < -transfer.RECAL_SLOPE_TOL],
        ["increasing (order not reversed)", "decreasing (order reversed)"],
        default=f"|slope| <= {transfer.RECAL_SLOPE_TOL:g} (effectively constant)")
    slope_signs = (_est.assign(direction=_dir)
                   .groupby(["family", "threshold", "direction"]).size()
                   .rename("cells").reset_index())
    print(f"\nfitted slope direction per cell, tolerance "
          f"{transfer.RECAL_SLOPE_TOL:g}")
    display(slope_signs.pivot_table(index=["family", "threshold"], columns="direction",
                                    values="cells", fill_value=0))
    print(f"  non-estimable mappings (no slope fitted): "
          f"{int((~RECAL_ESTIMATED).sum())}")

    # A non-zero slope can still map every score onto nearly the same probability.
    _flat = _est[_est["recal_score_range"] < 1e-6]
    print(f"  mappings whose corrected scores span < 1e-6: {len(_flat)}")

    # Raw against recalibrated AUC, paired within the cell. A positive slope means the
    # order is NOT REVERSED; it does not promise identical AUC, because `_lclip` bounds
    # the score before the logit and can tie values at the boundary.
    _key = ["family", "threshold", "seed"]
    _paired = (_est.set_index(_key)[["auc", "recal_slope", "recal_clipped_fraction"]]
               .rename(columns={"auc": "auc_recal"})
               .join(raw_focal.set_index(_key)["auc"].rename("auc_raw"), how="inner"))
    _paired["d_auc"] = _paired["auc_recal"] - _paired["auc_raw"]
    _inc = _paired[_paired["recal_slope"] > transfer.RECAL_SLOPE_TOL]
    print(f"\nAUC change from recalibration, {len(_paired)} paired estimable cells")
    if len(_inc):
        print(f"  increasing mappings: {len(_inc)}, largest |change| "
              f"{_inc['d_auc'].abs().max():.2e}, clipped fraction up to "
              f"{_inc['recal_clipped_fraction'].max():.4f}")
        print("  A change here is not automatically float noise: clipping ties scores "
              "at the bounds, and a material change is a finding to investigate.")
    _other = _paired[_paired["recal_slope"] <= transfer.RECAL_SLOPE_TOL]
    if len(_other):
        print(f"  non-increasing mappings: {len(_other)}, change range "
              f"{_other['d_auc'].min():+.4f} to {_other['d_auc'].max():+.4f} "
              f"— a reordering")

    print("\nfitted mapping, per cell rather than averaged")
    display(_est.groupby(["family", "threshold"])[
        ["recal_intercept", "recal_slope", "recal_clipped_fraction",
         "recal_score_range"]].agg(["min", "median", "max"]).round(6))

---
# J — Label budgets

**Question.** Section H spends exactly five hundred target outcomes. An authority facing that
cost has no reason to accept five hundred. Where do the returns to a larger budget flatten?

Two procedures swept from k = 50 to 2,000, one per cell. The draws are **nested**: the ordering
is built so that its first five hundred rows are the k=500 anchor Section H spends, every
smaller budget is a prefix of it and every larger one extends it. So each step is a within-seed
comparison and the k=500 column reconciles with Section H cell for cell.

**Nesting buys pairing, not monotonicity.** More outcomes can hurt, so a non-monotone step is a
result to read rather than a bug. Section B checks the nesting on the subset identifiers, never
on the metric.

**What is fitted.** The base is the canonical source model — the same fit, reused. The models
that vary with k cannot be: they are fitted on k target records and that is what the curve
measures. One fit per (family, threshold, seed, budget) per curve.

The unadapted reference does not vary with k and is not recomputed here. Section E already
holds it, and the assembly cell puts it beside the curves.

**Status: primary analysis.**

### Target-only across the budget

Discards the source model and trains on the k target outcomes alone. At small k this is the arm
with the least to work with; at large k it is what a target-side model reaches with that many
records.

In [ ]:
budget_target_only = transfer.label_budget_curve(
    splits, families=FAMILIES, thresholds=tuple(THRESHOLDS),
    regimes=("target_only",), budgets=transfer.BUDGETS,
    tuned=MCS_SETTINGS, sources=source_models)

if not PUBLIC_NOTEBOOK:
    print(f"budgets swept: {list(transfer.BUDGETS)}")
display(budget_target_only[budget_target_only["metric"] == "auc"]
        .pivot_table(index=["family", "threshold"], columns="k", values="mean").round(4))

### Full revision across the budget

Re-estimates every parameter of the source model: a warm start for the three families that
support one, otherwise a weighted refit on MCS plus the k target outcomes. The manuscript calls
this *full revision*; the pipeline key is `fine_tune`, and `label_budget_curve` translates at
this boundary.

In [ ]:
budget_full_revision = transfer.label_budget_curve(
    splits, families=FAMILIES, thresholds=tuple(THRESHOLDS),
    regimes=("full_revision",), budgets=transfer.BUDGETS,
    tuned=MCS_SETTINGS, sources=source_models)

display(budget_full_revision[budget_full_revision["metric"] == "auc"]
        .pivot_table(index=["family", "threshold"], columns="k", values="mean").round(4))

### The two curves together

The flat unadapted reference is scored here rather than taken from Section E, and the reason is
a difference in what is measured rather than in what is fitted. **No model is fitted**: the
canonical source models are scored, so the reference is the same models Section E reported. But
the curve reports precision and recall at 5% and 15% capacity — the Hello Baby and AFST
operating points — where the Section E battery reports the decile and the quintile. The
reference has to be expressed on the curve's own operating points for `budget_summary`'s
5%-precision delta to mean anything.

In [ ]:
# The flat reference: the canonical source models scored at every budget, on the curve's own
# 5% and 15% operating points. `sources=` means no model is fitted here.
budget_unadapted = transfer.label_budget_curve(
    splits, families=FAMILIES, thresholds=tuple(THRESHOLDS),
    regimes=("unadapted",), budgets=transfer.BUDGETS,
    tuned=MCS_SETTINGS, sources=source_models)

label_budget = pd.concat(
    [budget_unadapted, budget_target_only, budget_full_revision], ignore_index=True)
label_budget_summary = transfer.budget_summary(label_budget)

if not PUBLIC_NOTEBOOK:
    print(f"{len(label_budget):,} rows | {label_budget.regime.nunique()} regimes x "
          f"{label_budget.k.nunique()} budgets")
display(label_budget_summary)

---
# K — Calibration correction

**Question.** Sections H and I each spend the whole budget on one thing — updating the model, or
correcting its probabilities. What happens in between?

Four post-hoc corrections, three regimes, and four ways of dividing the five hundred records
between updating and calibrating. One cell per regime; the ratios and the corrections are named
in each cell rather than chosen inside a dispatcher.

| | |
|---|---|
| **corrections** | `none`, `logistic`, `isotonic`, `beta` — every one monotone, so AUC and PR-AUC carry as a monotonicity check against the `none` column |
| **ratios** | (500, 0), (400, 100), (350, 150), (300, 200) — update records first, calibration records second |
| **the 500/0 arm** | has no held-out fold, so the correction is fitted on the records the model was updated on. It is an optimistic baseline, kept because it makes the overfitting cost visible rather than hiding it |
| **slope and intercept** | meaningful for `none` and `logistic` only; the grid stays rectangular and carries NaN elsewhere with its reason |

**What is fitted.** One base per (family, threshold, seed) — the canonical source model, reused —
plus one update fit per ratio for each label-using regime. The update fits are fitted on
different folds of the anchor and cannot be shared.

**Status: primary analysis.**

### Correcting the unadapted baseline

The model is left alone and only its probabilities are corrected. The whole budget goes to the
correction, so `update_n` is zero at every ratio.

In [ ]:
sweep_unadapted = transfer.calibration_correction_sweep(
    splits, families=FAMILIES, thresholds=THRESHOLDS, seeds=SEEDS,
    tuned=MCS_SETTINGS, sources=source_models, regimes=("unadapted",))

print(f"corrections {list(transfer.CORRECTIONS)} x ratios {list(transfer.RATIOS)}")
display(sweep_unadapted[sweep_unadapted["metric"] == "ece"]
        .pivot_table(index=["family", "threshold"], columns=["correction", "calib_n"],
                     values="mean").round(4))

### Correcting a target-only model

The update fold trains a target-side model from scratch; the calibration fold corrects it. At
(500, 0) both are the same records.

In [ ]:
sweep_target_only = transfer.calibration_correction_sweep(
    splits, families=FAMILIES, thresholds=THRESHOLDS, seeds=SEEDS,
    tuned=MCS_SETTINGS, sources=source_models, regimes=("target_only",))

display(sweep_target_only[sweep_target_only["metric"] == "ece"]
        .pivot_table(index=["family", "threshold"], columns=["correction", "calib_n"],
                     values="mean").round(4))

### Correcting a fully revised model

The update fold revises every parameter of the source model; the calibration fold corrects what
comes out.

In [ ]:
sweep_full_revision = transfer.calibration_correction_sweep(
    splits, families=FAMILIES, thresholds=THRESHOLDS, seeds=SEEDS,
    tuned=MCS_SETTINGS, sources=source_models, regimes=("full_revision",))

display(sweep_full_revision[sweep_full_revision["metric"] == "ece"]
        .pivot_table(index=["family", "threshold"], columns=["correction", "calib_n"],
                     values="mean").round(4))

### The three regimes together

In [ ]:
calibration_sweep = pd.concat(
    [sweep_unadapted, sweep_target_only, sweep_full_revision], ignore_index=True)

# Validated against the table's own declaration rather than a hand-rolled column list.
_problems = paper.validate(paper.BY_KEY["calibration"], calibration_sweep)
if _problems:
    raise ValueError("the calibration sweep fails its declaration:\n  "
                     + "\n  ".join(_problems))
if not PUBLIC_NOTEBOOK:
    print(f"{len(calibration_sweep):,} rows, valid against the `calibration` declaration")

display(calibration_sweep[calibration_sweep["metric"] == "ece"]
        .groupby(["regime", "correction", "calib_n"])["mean"].median().round(4)
        .unstack("correction"))

In [ ]:
# The §VI-D cell the write-up quotes: baseline x logistic x >=2 at 100 calibration records.
cell = calibration_sweep.query("regime == 'unadapted' and correction == 'logistic' and threshold == '>=2' "
                   "and calib_n == 100 and metric == 'ece'")
if cell.empty:
    raise ValueError("the §VI-D slice selected no rows; the sweep's scope has changed")
dupes = cell[cell.duplicated("family", keep=False)]["family"].unique()
if len(dupes):
    raise ValueError(f"duplicate family rows at this cell: {sorted(dupes)}")

# The range and the comparison with the earlier run are commentary rather than result,
# and the seed column is protocol. The per-family ECE is what §VI-D quotes, and it is
# what stays.
if not PUBLIC_NOTEBOOK:
    print(f"this run: ECE {cell['mean'].min():.3f} to {cell['mean'].max():.3f} "
          f"at calib_n = 100")
    print("the published sweep gave 0.060 to 0.077 at the same cell — an observation")
    print("from the run the manuscript was written from, not a threshold this run has")
    print("to meet. A difference is a finding to report.")
display(cell[["family", "mean", "sd"]].sort_values("mean"))

---
# L — Structure transfer

**Question.** Where a tree's *parameters* do not carry across, does its *structure* — the splits
it learned, the leaves it defines — carry across on its own?

The repository does not record when these three mechanisms were chosen or what prompted them.
The manuscript reports leaf refresh in §V-F and the archived pipeline implemented all three.
**This section is therefore a transparent reconstruction of where the experiment belongs in the
argument, not a record of when it was run.** As placed here it takes up a question Section H
raises: whether the tree families fail to transfer for a different reason from the linear ones.

Three mechanisms — leaf refresh, rule head, leaf membership — on the same protocol as everything
above, so they are comparable with the rest of the notebook rather than only with each other.
Status: exploratory follow-up.

**This is where the manuscript's leaf-refresh numbers come from.** Section H's
`leaf_refresh_global` is a different thing despite the similar name: one global intercept offset,
not a relearned leaf value.

**Scope: tree families only.** The three mechanisms have no cross-family analogue for the linear
families, so LR rows are absent.

### Leaf refresh

The tree structure is held and the leaf values are re-estimated on the target outcomes. **This
is the mechanism the manuscript reports in §V-F.** Section H's `leaf_refresh_global` is a
different thing despite the name: one global intercept offset, not a relearned leaf value.

In [ ]:
leaf_refresh = transfer.run_backfill(
    ("leaf_refresh",), splits=splits, thresholds=tuple(THRESHOLDS),
    tuned=MCS_SETTINGS, sources=source_models)
if not PUBLIC_NOTEBOOK:
    print(f"{len(leaf_refresh)} rows | family "
          f"{sorted(leaf_refresh.family.unique())}")

### Rule head

The splits the tree learned are extracted as rules and a sparse linear head is fitted over them
on the target outcomes. What carries across is the rule set, not the coefficients.

In [ ]:
rule_head_transfer = transfer.run_backfill(
    ("rule_head",), splits=splits, thresholds=tuple(THRESHOLDS),
    tuned=MCS_SETTINGS, sources=source_models)
if not PUBLIC_NOTEBOOK:
    print(f"{len(rule_head_transfer)} rows | family "
          f"{sorted(rule_head_transfer.family.unique())}")

### Leaf membership

Each respondent is represented by which leaves they fall into, and a head is fitted on that
representation. The structure is used as a feature map rather than as a predictor.

In [ ]:
leaf_membership = transfer.run_backfill(
    ("leaf_membership",), splits=splits, thresholds=tuple(THRESHOLDS),
    tuned=MCS_SETTINGS, sources=source_models)
if not PUBLIC_NOTEBOOK:
    print(f"{len(leaf_membership)} rows | family "
          f"{sorted(leaf_membership.family.unique())}")

### The three mechanisms together

In [ ]:
structure_transfer = transfer.consolidate_backfill(
    pd.concat([leaf_refresh, rule_head_transfer, leaf_membership], ignore_index=True))

display(structure_transfer[["family", "method", "arm", "threshold", "auc_mean", "auc_sd",
                            "prauc_mean", "ece_mean"]]
        .set_index(["family", "method", "arm", "threshold"]).round(4))
if not PUBLIC_NOTEBOOK:
    print(f"{len(structure_transfer)} rows over "
          f"{structure_transfer.method.nunique()} mechanisms, "
          f"{structure_transfer.family.nunique()} families")

---
# M — Outcome robustness

## Leave one pillar out

**Question.** Does any of the above depend on how the outcome was defined?

The outcome is a count over five welfare pillars, and a count is a modelling choice. A result
that survives every leave-one-out is a result about welfare risk; one that disappears when a
pillar is removed is a result about that pillar. Each pillar is dropped in turn and the
comparison re-run on the remaining four.

**The partition is held fixed, and that is the load-bearing detail.** All six variants are
evaluated on the **same** MCS-test and YRBS-test partition for a given seed — the `>=1` bundle
stratified on the full-five outcome, which Section B already built. Only the label definition
varies, which is what the decomposition requires. Stratifying each variant on its own outcome
would confound "the pillar matters" with "the split moved".

**This is the opposite convention from the outcome-variant battery below**, which does stratify
each variant on its own outcome. The two answer related but different sensitivity questions and
are not interchangeable. Neither is changed here; Section N carries the difference.

**Scope: three families** — L1_LR, XGB and CatBoost. That is the scope the experiment was run at,
and **why these three** is one of Section N's open questions. A >=1-only analysis: a
leave-one-out outcome at >=2 over four pillars is a different construct, not a threshold variant.

**Status: robustness.**

In [ ]:
# The six outcomes, composed here rather than inside the battery. `full` is the five-pillar
# outcome at >=1 — the same series Section B built, and the one every variant's split is
# stratified on.
loo_outcomes = {
    "full": (data.make_outcome(pillars_mcs, 1), data.make_outcome(pillars_yrbs, 1)),
}
for variant, pillar in transfer.LOO_PILLARS.items():
    loo_outcomes[variant] = (
        outcomes.compose_outcome_loo(pillars_mcs[list(data.SHARED_PILLARS)], pillar,
                                     threshold=1, strict=True),
        outcomes.compose_outcome_loo(pillars_yrbs[list(data.SHARED_PILLARS)], pillar,
                                     threshold=1, strict=True))

LOO_FAMILIES = ("L1_LR", "XGB", "CatBoost")
if not loo_outcomes["full"][0].equals(y_mcs[1]):
    raise ValueError("the leave-one-out stratifier is not the >=1 outcome the splits were "
                     "built on")
print(f"{len(loo_outcomes)} outcomes: {', '.join(loo_outcomes)}")
print("shared partition: splits[(seed, 1)], stratified on the full-five outcome")
print(f"families: {', '.join(LOO_FAMILIES)}")

### Dropping nothing — the five-pillar reference

The reference every drop is read against: the full five-pillar outcome at >=1. Its fit is the canonical source model, reused rather than refitted.

In [ ]:
loo_full = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("full",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_full.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Dropping sexual abuse

Fits a new model: the label has changed, so the source model has too.

In [ ]:
loo_sexual = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("loo_sexual",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_sexual.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Dropping emotional abuse

Fits a new model on the remaining four pillars.

In [ ]:
loo_emotional = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("loo_emotional",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_emotional.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Dropping physical abuse

Fits a new model on the remaining four pillars.

In [ ]:
loo_physical = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("loo_physical",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_physical.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Dropping household mental illness

Fits a new model on the remaining four pillars.

In [ ]:
loo_mental = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("loo_mental",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_mental.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Dropping household substance use

Fits a new model on the remaining four pillars.

In [ ]:
loo_substance = transfer.leave_one_pillar_out(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("loo_substance",),
    splits=splits, sources=source_models)
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(loo_substance.groupby("model")[["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### The six variants together

Assembled from the six named frames, after all six have run. The per-seed frame is kept because
notebook 03 reads it; what is displayed is the across-seed comparison, which is what the
sensitivity question needs.

In [ ]:
loo_sensitivity = pd.concat(
    [loo_full, loo_sexual, loo_emotional, loo_physical, loo_mental, loo_substance],
    ignore_index=True)

if not PUBLIC_NOTEBOOK:
    print(f"{len(loo_sensitivity):,} per-seed rows | "
          f"{loo_sensitivity['variant'].nunique()} variants x "
          f"{loo_sensitivity['model'].nunique()} families x "
          f"{loo_sensitivity['seed'].nunique()} seeds")

display(loo_sensitivity
        .groupby(["variant", "dropped_pillar", "model"])[["mcs_auc_internal",
                                                          "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

## Outcome variants

**Question.** The same question at wider range: does the result depend on the five-pillar
construct at all?

Every symmetric variant of the outcome — the two category clusters and four single pillars —
across twenty seeds, at the same three-family scope. A variant whose MCS and YRBS compositions
differ is not run at all: it would train on one construct and evaluate on another, and the
symmetry check halts rather than warning.

**Each variant gets its own partition**, stratified on its own outcome. That is the opposite of
the leave-one-out convention above, where one partition is held fixed. Whether these two should
be brought into line is a methodological decision and not a structural one; Section N carries it.

Some variants are not usable — `pillar_sexual` trips a prevalence guard, `felitti_neglect` has
no MCS pillar, several are cohort-asymmetric — and those are recorded with the reason rather
than quietly dropped. A >=1-only analysis: these are already narrower constructs, and a >=2 cut
on a two-pillar cluster is the conjunction rather than a threshold sweep.

**Status: robustness.**

In [ ]:
# The six symmetric variants, named here rather than discovered inside the battery. Each is a
# subset of the five shared pillars, composed strictly at >=1.
VARIANT_OUTCOMES = dict(transfer.E10_OUTCOMES)
for _name, _key in VARIANT_OUTCOMES.items():
    print(f"  {_name:<24} {_key:<20} {', '.join(outcomes.CATEGORY_SETS[_key])}")

print("\nexcluded, with the reason recorded rather than the variant dropped:")
print("  pillar_sexual        prevalence guard (<0.05)")
print("  felitti_neglect      no MCS pillar exists")
print("  hughes_* variants    cohort-asymmetric composition")
print("  flat-5 >=4           prevalence guard")

### The abuse cluster

Sexual, emotional and physical abuse taken together. Its own partition, stratified on this outcome.

In [ ]:
variant_abuse_cluster = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("abuse_cluster",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_abuse_cluster.pivot_table(index="model", columns="role", values="auc").round(4))

### The household cluster

Household substance use and household mental illness taken together. Its own partition, stratified on this outcome.

In [ ]:
variant_household_cluster = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("household_cluster",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_household_cluster.pivot_table(index="model", columns="role", values="auc").round(4))

### Emotional abuse alone

One pillar as the whole outcome. Its own partition, stratified on this outcome.

In [ ]:
variant_pillar_emotional = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("pillar_emotional",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_pillar_emotional.pivot_table(index="model", columns="role", values="auc").round(4))

### Physical abuse alone

One pillar as the whole outcome. Its own partition, stratified on this outcome.

In [ ]:
variant_pillar_physical = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("pillar_physical",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_pillar_physical.pivot_table(index="model", columns="role", values="auc").round(4))

### Household mental illness alone

One pillar as the whole outcome. Its own partition, stratified on this outcome.

In [ ]:
variant_pillar_mental = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("pillar_mental",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_pillar_mental.pivot_table(index="model", columns="role", values="auc").round(4))

### Household substance use alone

One pillar as the whole outcome. Its own partition, stratified on this outcome.

In [ ]:
variant_pillar_substance = transfer.outcome_variant_battery(
    pillars_mcs, pillars_yrbs, X_mcs=X_mcs, X_yrbs=X_yrbs, seeds=SEEDS,
    families=LOO_FAMILIES, tuned=MCS_SETTINGS, variants=("pillar_substance",))
if not PUBLIC_NOTEBOOK:                 # the combined table is below
    display(variant_pillar_substance.pivot_table(index="model", columns="role", values="auc").round(4))

### The six variants together

Assembled after all six have run. Notebook 03 reads the per-seed frame; the display is the
across-seed comparison.

In [ ]:
outcome_variants = pd.concat(
    [variant_abuse_cluster, variant_household_cluster, variant_pillar_emotional,
     variant_pillar_physical, variant_pillar_mental, variant_pillar_substance],
    ignore_index=True)

if not PUBLIC_NOTEBOOK:
    print(f"{len(outcome_variants):,} per-seed rows | "
          f"{outcome_variants['outcome'].nunique()} outcome variants x "
          f"{outcome_variants['model'].nunique()} families x "
          f"{outcome_variants['seed'].nunique()} seeds")

display(outcome_variants.pivot_table(index=["outcome", "model"], columns="role",
                                     values="auc", aggfunc="mean").round(4))

---
# N — Limitations, and design questions that remain open

Flagged rather than fixed. Each would change the estimand, the compute, or both, and none is a
decision to take inside a structural pass.

**1. Both local references are internal-development estimates.** Each was developed and
evaluated inside one cohort, so neither is independent external validation. Both
configurations come from the SAME three-seed consensus procedure — development seeds 0, 1
and 2, five-fold stratified inner cross-validation, AUC, the mean of the three seed-level
means — run inside each cohort's own outer-training partitions.

Development and evaluation resamples overlap within each cohort: respondents held out in
one evaluation split may appear in development partitions under other seeds, so both
references may retain model-development optimism. **Applying the same procedure to both
aligns them; it does not establish that the magnitude of any optimism is identical.**
Within any inner fold no held-out record contributed to the fitting half's model or its
transformation — the overlap arises from repeated outer resampling, not within-fold
leakage.

The across-split standard deviation describes sensitivity to partitioning and **is not a
standard error**. Held-out fold covariates and every evaluation frame are standardised
against themselves, so the transformation is **transductive within the covariate batch**
rather than inductive. Harmonising the predictors and the outcome across cohorts does not
establish measurement equivalence, and no claim of external generalisability follows.

**1a. Development and evaluation resampling overlap, in both cohorts.** Each cohort's configuration
is selected on development seeds 0, 1 and 2, inside those seeds' *training* partitions, and all
twenty evaluation seeds reuse it. Seeds 3 to 19 draw their test partitions independently from the
same cohort, so a large share of each one's test respondents sat in a development training
partition and contributed labels to the cross-validation that chose the configuration; those
within-cohort estimates are mildly optimistic. Seeds 0, 1 and 2 are the cleaner case: their own
held-out records are disjoint from their own development data by construction.

**Both local references carry this, and they carry it symmetrically.** `mcs_internal` and
`yrbs_local` are produced by the same procedure applied inside each cohort, so the optimism is a
property of the protocol rather than of one side of the comparison. It does not follow that its
magnitude is equal in the two cohorts. `transfer_loss` is measured down from `mcs_internal` and
`target_resource_gap` up to `yrbs_local`, so an inflated reference makes each of them larger —
which runs against the finding that discrimination degrades across the border, not toward it.

**Neither selection reads the other cohort's data.** No YRBS frame, index or label enters the MCS
selection and no MCS frame enters the YRBS one, and no outer test partition of either cohort
enters either. `unadapted` and every adapted regime are configured by the MCS mapping alone.

*Recommendation, not applied:* nested selection inside each evaluation seed's training partition
removes the overlap, at a large multiple of the present fitting cost, and changes the estimand
from "one fixed configuration evaluated twenty times" to "a selection procedure evaluated twenty
times". The paper makes the first claim. `transfer.nested_target_sensitivity_scores` implements
exactly that on the target side and is off by default. **Pairing would survive either way** —
comparisons are within-seed, and a seed-specific configuration shared by both members of a pair
leaves the pairing intact.

**1b. Every transfer procedure remains conditional on the fixed MCS configuration.** The
target resource gap is the distance from unadapted transfer up to a model developed inside
YRBS by the same procedure, which is what an importing authority building its own model
would face.

**2. The twenty splits overlap each other.** They share most of their training rows and about a
quarter of each test set, so the twenty results are positively correlated and the across-seed sd
understates the variability of a fresh sample. *Read it as a stability measure, not a standard
error.*

**3. No inferential test is reported.** The twenty paired differences per (family, threshold)
are not independent, for the reason above, so the Wilcoxon signed-rank test's assumptions are
not met and its nominal level would not be attained. Signed-rank p-values, their Holm
adjustment and the Hodges-Lehmann interval have been removed from this notebook rather than
reported with a caveat. What remains is the mean paired difference across the twenty shared
splits and its spread, which the design does support. *A confidence interval that corresponds
to the reported means is a separate design question and is not answered here.*

**4. The reporting groups are declared, not derived.** §V-D justifies separating probability
adjustment from adaptation, and that is implemented. Splitting label-free from label-using is
justified nowhere in this repository or the draft. With no test reported the grouping no longer
changes any number, but it still organises what is compared with what. **Rationale to
document.**

**5. k = 500 has no recoverable derivation.** The draft calls it "a comparison point, not a
general sufficiency threshold" and derives the value nowhere; no power calculation or cost model
is in this repository. Section J's curves make the choice legible without justifying it.
**Rationale to document.**

**6. The robustness scope is three families and no committed file says why.** L1_LR, XGB and
CatBoost span the three model classes, which is a plausible reconstruction and not a record.
**Rationale to document.**

## Methodological cautions that stand

**7. Test-frame standardisation is transductive.** How a target test row is transformed depends
on the other target test rows. Between cohorts that is the reported adaptation; within one cohort
it means both reference arms fit in training units and predict in test units. It is fitted in Section B, by `data.build_splits`, and not upstream. Section C and
`FINDING_cohort_standardisation.md`.

**8. The evaluable sample is smaller than the test frame.** `n_test` is the evaluable count; the
frame's row count is larger. Both are correct.

**9. The target-label pool and the test set are disjoint by construction**, and the one way that
could break is a duplicated index label. Section B asserts it on every bundle.

**10. The budget draws are nested, which buys pairing and not monotonicity.**

**11. Discrimination and calibration are reported separately and never averaged.** A single
summary number would hide the one thing Section E is about.

**12. One arm has no transfer results.** Rung 1 of Section C's ladder is not computed at
all, so it cannot be compared with anything after transfer, and nothing reads it as though
it could.
**13. The two outcome-robustness batteries use different split conventions.** Leave-one-pillar-out
holds one full-five `>=1` partition fixed across all six variants, so only the label definition
moves. The outcome-variant battery stratifies each variant on its own outcome, so the partition
moves with the construct. The first isolates the effect of the label; the second measures each
construct on the sample that construct defines. They answer related but different questions and
their numbers should not be read as one series. **Whether they should be brought into line is a
methodological decision and was not taken during a structural pass.**

**14. The nested per-split target redevelopment is not run here.** A fresh YRBS selection
inside each of the twenty training partitions is a separate sensitivity analysis. It
supplies no headline reference, no headline gap, no handoff row and no published table,
and this notebook neither runs nor loads it. Whether a fully nested redevelopment would
reach somewhere different from the fixed YRBS local specification is undetermined here,
not settled.

---
# O — Summary and outputs

Every experiment above has now run and holds its result in a named dataframe. This section
concatenates the regime frames once, checks that every cell carries every seed, and writes the
four tables and one score file that have a downstream reader.

In [ ]:
# Collect every result frame produced in this run.
result_frames = [
    references,
    unadapted,
    quantile_mapping,
    importance_weighting,
    pseudo_labelling,
    platt_recalibration,
    isotonic_recalibration,
    intercept_offset,
    coefficient_freezing,
    sign_support,
    source_feature_set,
    ensemble_same_family,
    ensemble_catboost_source,
    threshold_self_training,
    target_only,
    full_revision,
    raw_l1_head,
    prior_correction,
    recalibrated_focal,
]

if source_scaled is not None:
    result_frames.append(source_scaled)

all_results = pd.concat(result_frames, ignore_index=True)


# Summarise the seed-level results before calculating reference-relative quantities.
all_summary = evaluation.summarise_seeds(all_results)


# Add the recalibration estimability information.
RECAL_REGIMES = [
    f"{regime}_logistic_recal"
    for regime, _ in transfer.FOCAL_PIPELINES
]

all_summary = all_summary.merge(
    recal_estimability,
    on=["regime", "family", "threshold"],
    how="left",
)

not_recalibrated = ~all_summary["regime"].isin(RECAL_REGIMES)
all_summary.loc[
    not_recalibrated,
    "recalibration_estimability_status",
] = "not applicable"
all_summary.loc[not_recalibrated, "non_estimable_reasons"] = ""


# Do not report a mean over only the seeds on which recalibration was estimable.
INCOMPLETE_RECAL = (
    all_summary["recalibration_estimability_status"] == "incomplete"
)

if INCOMPLETE_RECAL.any():
    metric_columns = [
        column
        for column in all_summary.columns
        if column.endswith(("_mean", "_sd", "_plo", "_phi"))
    ]

    all_summary.loc[INCOMPLETE_RECAL, metric_columns] = np.nan

    print(
        f"{int(INCOMPLETE_RECAL.sum())} recalibration cell(s) are incomplete. "
        "Their across-seed performance summaries are blank; the per-seed "
        "battery retains every estimated result."
    )


# Calculate the reference gaps only after incomplete performance summaries
# have been blanked.
all_summary = evaluation.add_reference_gaps(all_summary)


# These quantities depend on the current procedure's AUC and must therefore
# also be blank for an incomplete recalibration cell.
procedure_dependent_gaps = [
    "transfer_loss",
    "adaptation_gain",
    "target_gap_recovered",
]

missing_gap_columns = [
    column
    for column in procedure_dependent_gaps
    if column not in all_summary.columns
]
if missing_gap_columns:
    raise ValueError(
        "add_reference_gaps did not produce the expected columns: "
        f"{missing_gap_columns}"
    )

incomplete_gap_values = all_summary.loc[
    INCOMPLETE_RECAL,
    procedure_dependent_gaps,
]

if incomplete_gap_values.notna().any().any():
    raise ValueError(
        "an incomplete recalibration cell carries a "
        "procedure-dependent reference result"
    )


# Check that every declared experimental cell contains all seeds.
audit = (
    all_results
    .groupby(["regime", "family", "threshold"])["seed"]
    .nunique()
    .rename("seeds")
    .reset_index()
)

short_cells = audit[audit["seeds"] != len(SEEDS)]
if len(short_cells):
    raise RuntimeError(
        f"{len(short_cells)} cell(s) are short of {len(SEEDS)} seeds:\n"
        f"{short_cells.to_string(index=False)}"
    )

display(
    audit
    .groupby("regime")
    .agg(
        cells=("family", "size"),
        families=("family", "nunique"),
        seeds=("seeds", "min"),
    )
    .sort_index()
)

print(
    f"\n{all_results['regime'].nunique()} regimes | "
    f"{len(all_results):,} metric rows | "
    f"every cell carries {len(SEEDS)} seeds"
)

### The person-level predictions this run keeps

Five named frames, each narrowed where it was produced. Every row is a YRBS evaluation
prediction; MCS predictions are excluded at source by `transfer.metric_rows` and cannot reach
this frame.

The unique key is `(threshold, family, regime, seed, row_id)`. `regime` distinguishes a
recalibrated pipeline from its raw counterpart, so no separate flag column is needed, and `arm`
is dropped because it is constant.

Fourteen hundred and forty cells is not the count: the scope is nine unadapted families, nine
target-trained references and the four focal pipelines twice over — raw and recalibrated —
across three thresholds and twenty seeds.

In [ ]:
# The person-level YRBS predictions this run keeps, named frame by named frame. Every one was
# narrowed at its own experiment cell; nothing is selected here.
yrbs_scores = pd.concat(
    [unadapted_scores,            # nine families
     reference_scores,            # yrbs_local, nine families
     target_only_scores,          # focal: L1_LR
     full_revision_scores,        # focal: RF, HistGB, CatBoost
     recalibrated_focal_scores],  # the same four, after cross-fitted recalibration
    ignore_index=True)

SCORE_COLUMNS = ["threshold", "family", "regime", "seed", "row_id", "y_true", "score"]
SCORE_KEY = ["threshold", "family", "regime", "seed", "row_id"]

# `arm` is constant "tuned" for every score-bearing regime, so it carries no information and is
# dropped. No role or recalibration column: every row here is an evaluation prediction, and the
# regime name already tells a recalibrated row from a raw one.
yrbs_scores = yrbs_scores[SCORE_COLUMNS]

EXPECTED_SCORE_SCOPE = (
    {("unadapted", f) for f in FAMILIES}
    | {("yrbs_local", f) for f in FAMILIES}
    | {(reg, fam) for reg, fam in transfer.FOCAL_PIPELINES}
    | {(f"{reg}_logistic_recal", fam) for reg, fam in transfer.FOCAL_PIPELINES})

# THE SCOPE MAY BE A SUBSET, because a recalibration cell that could not be estimated
# writes no person-level rows by design. What may not happen is a pair outside the
# declared set. The per-cell accounting below is what has to add up.
_scope = set(map(tuple, yrbs_scores[["regime", "family"]].drop_duplicates().to_numpy()))
if not _scope <= EXPECTED_SCORE_SCOPE:
    raise ValueError(f"undeclared (regime, family) pair(s) in the handoff: "
                     f"{sorted(_scope - EXPECTED_SCORE_SCOPE)}")
if "mcs_internal" in set(yrbs_scores["regime"]):
    raise ValueError("an MCS-evaluated regime reached the handoff")
if set(yrbs_scores["threshold"]) != {f">={t}" for t in THRESHOLDS}:
    raise ValueError("the handoff does not cover the three thresholds")
if set(yrbs_scores["seed"]) != set(SEEDS):
    raise ValueError("the handoff does not cover the twenty seeds")
if yrbs_scores.duplicated(SCORE_KEY).any():
    raise ValueError("a respondent appears twice in one scientific cell of the handoff")

# DECLARED, SCORE-BEARING AND NON-ESTIMABLE. A recalibration cell that could not be estimated
# writes no person-level rows by design, so requiring every declared cell to be score-bearing
# would fail on a legitimate outcome. What must hold is that the two account for all of them.
_cell_keys = ["regime", "family", "threshold", "seed"]
declared_keys = {(r, f, f">={t}", s) for r, f in EXPECTED_SCORE_SCOPE
                 for t in THRESHOLDS for s in SEEDS}
scoring_keys = set(map(tuple, yrbs_scores[_cell_keys].drop_duplicates().to_numpy()))
non_estimable_keys = set(map(tuple, recalibrated_focal.loc[
    recalibrated_focal["recal_status"] != "estimated", _cell_keys].to_numpy()))
# Recalibration is the ONLY procedure here that can fail to be estimated. Both local
# references take a fixed configuration and are estimated on every split.

declared_cells = len(declared_keys)
scoring_cells = len(scoring_keys)
non_estimable_cells = len(non_estimable_keys)

# KEYS, NOT COUNTS. Every declared cell is either score-bearing or has exactly one matching
# non-estimable metric row, on the same regime, family, threshold and seed. Checking totals
# alone would let an absent score cell be offset by an unrelated non-estimable one.
if not scoring_keys <= declared_keys:
    raise ValueError("score rows fall outside the declared scope")
if not non_estimable_keys <= declared_keys:
    raise ValueError("non-estimable rows fall outside the declared scope")
if scoring_keys & non_estimable_keys:
    raise ValueError("a cell is both score-bearing and recorded non-estimable")
_unaccounted = declared_keys - scoring_keys - non_estimable_keys
if _unaccounted:
    raise ValueError(f"{len(_unaccounted)} declared cell(s) produced neither scores nor a "
                     f"non-estimable record")

print(f"{declared_cells} declared scientific cells "
      f"({len(EXPECTED_SCORE_SCOPE)} regime-family pairs x {len(THRESHOLDS)} thresholds x "
      f"{len(SEEDS)} seeds)")
print(f"  {scoring_cells} score-bearing | {non_estimable_cells} non-estimable recalibration "
      f"cell(s), which write no person-level rows by design")

### What is written

Five files, and each has a reader. Everything else this notebook computed stays in memory,
because nothing reads it: a table written for no consumer is a claim that something depends on
it.

| file | grain | who reads it |
|---|---|---|
| `regime_battery.csv` | **one row per regime x family x arm x threshold x seed** | notebook 04 |
| `regime_battery_summary.csv` | across-seed mean, sd and percentiles | notebook 03, through `evaluation.regime_grid` |
| `loo_sensitivity_summary.csv` | **one row per variant x family x seed — per-seed rows, not a summary** | notebook 03 |
| `outcome_variants_summary.csv` | **one row per outcome x role x family x seed — per-seed rows, not a summary** | notebook 03 |
| `yrbs_scores.parquet` | one row per YRBS test respondent per seed, per regime — see below | notebook 03 |

NO PER-SEED TARGET SELECTION RECORD IS WRITTEN. The YRBS local reference takes one fixed
configuration from `spec/local_model_settings.csv`, promoted after review, so there is no
per-seed search outcome to record. The cross-validated values behind that selection live
in the private records under the working root and are not written here.

**What the score file carries.** YRBS evaluation predictions only, at all three thresholds and
all twenty seeds, for thirty-five (regime, family) pairs:

| | |
|---|---|
| `unadapted` | all nine families |
| `yrbs_local` | all nine families |
| `target_only`, `fine_tune` | the four focal pipelines — L1_LR, and RF / HistGB / CatBoost |
| `target_only_logistic_recal`, `fine_tune_logistic_recal` | the same four, after cross-fitted logistic recalibration |

MCS predictions are excluded at source and cannot reach it. A recalibration cell that could
not be estimated writes no rows, so the file's cell count is the declared count less those,
accounted for through `recal_status`.

The two files whose names say *summary* hold per-seed rows. The names are notebook 03's, which
reads them by name, so correcting them means changing producer and consumer together —
recorded as a notebook 03 handoff issue rather than done here.

**The MCS-derived columns that do not travel.** Every metric row carries the confusion counts
behind its precision, recall and specificity. For rows evaluated inside MCS those counts, and
the denominators beside them, are exact figures about restricted records; they are used in memory
to compute the rates and blanked before either battery file is written. Nothing downstream reads
them: notebook 04 takes AUC, and `evaluation.SUMMARY_METRICS` carries rates only.

In [ ]:
MCS_EXACT_COUNT_COLUMNS = ("dec_TP", "dec_FP", "dec_FN", "dec_TN",
                           "qui_TP", "qui_FP", "qui_FN", "qui_TN", "n_pos", "n_test")


def without_mcs_exact_counts(frame, mcs_rows):
    """Blank the exact MCS cell counts and denominators before a frame is written.

    This closes ONE disclosure route — recovering a cell count from the file — and nothing
    more. **It is not clearance.** Every MCS-derived aggregate on these rows, including the
    rates, the calibration statistics and `prevalence`, remains subject to disclosure review
    before it may leave the approved environment.

    `prevalence` stays because it is the PR-AUC null and a PR-AUC is unreadable without it;
    with the denominator gone it does not give a count back.
    """
    out = frame.copy()
    cols = [c for c in MCS_EXACT_COUNT_COLUMNS if c in out.columns]
    out.loc[mcs_rows, cols] = np.nan
    return out


# The two battery files carry MCS-derived aggregate rows at every threshold and are therefore
# RESTRICTED BY CONTENT. They are written for local inspection and for notebook 03 inside the
# approved environment. Nothing here approves anything for external reporting, and no such
# approval follows from a directory, a filename or a structural check.
inputs.save_table(
    without_mcs_exact_counts(all_results, all_results["is_mcs"].astype(bool)),
    "regime_battery.csv")                       # per-seed rows; notebook 04 reads it
inputs.save_table(
    without_mcs_exact_counts(all_summary, all_summary["regime"] == "mcs_internal"),
    "regime_battery_summary.csv")               # across-seed summary; notebook 03 reads it
inputs.save_table(loo_sensitivity, "loo_sensitivity_summary.csv")       # per-seed; notebook 03
inputs.save_table(outcome_variants, "outcome_variants_summary.csv")     # per-seed; notebook 03

# The target-side search's record: one row per (family, threshold, seed), written AFTER the
# search ran. It is an OUTPUT of this notebook and never an input to it — nothing above reads
# it, and a stale copy from an earlier run cannot influence anything here. Notebook 03 reads it
# to render the benchmark's appendix block.

# The one person-level artefact, under the secure root, written once. It is a computational
# input to notebook 03, not a publication file, and it is never tracked.
scores.write_yrbs_scores(yrbs_scores, quiet=True)
print("  person-level YRBS scores written to the configured restricted location")

In [ ]:
# Computed from the frames just written; nothing recalled from a previous run or the manuscript.
print("SUMMARY OF THIS RUN")
print(f"  {all_results['regime'].nunique()} regimes x {all_results['family'].nunique()} families "
      f"x {all_results['threshold'].nunique()} thresholds x {all_results['seed'].nunique()} seeds")
print(f"  {len(source_models)} canonical source models, fitted once in Section D and reused")
print(f"  outputs: regime_battery.csv, regime_battery_summary.csv, "
      f"loo_sensitivity_summary.csv, outcome_variants_summary.csv, "
      f"yrbs_scores.parquet")

_u = all_summary[all_summary["regime"] == "unadapted"]
print(f"\ntransfer, over {len(_u)} (family, threshold) cells:")
print(f"  median AUC {_u['auc_mean'].median():.4f}   "
      f"median transfer loss against the MCS local reference "
      f"{_u['transfer_loss'].median():+.4f}")

# The target resource gap is cell-level, so it is the same on every row of a cell and is
# read off the unadapted row here.
_gaps = _u[["target_resource_gap"]].dropna()
if len(_gaps):
    print(f"\nthe target resource gap, over {len(_gaps)} cells:")
    print(f"  median {_gaps['target_resource_gap'].median():+.4f}")
_missing = _u[_u["target_gap_reason"] != ""]
if len(_missing):
    print(f"  {len(_missing)} cell(s) carry no target resource gap; reasons: "
          f"{sorted(_missing['target_gap_reason'].unique())}")

# `scores.LABEL_USING` is Section H's twelve-procedure mechanism comparison. The two
# recalibrated focal regimes are label-using but are NOT members of it — they are their own
# reporting group, run over four pipelines rather than nine families — so they are reported
# separately rather than folded in, which would change what the comparison compares.
_best = (all_summary[all_summary["regime"].isin(scores.LABEL_USING)]
         .groupby("regime")["auc_mean"].median().sort_values(ascending=False))
print(f"\nhighest median AUC among Section H's {len(_best)} label-using procedures "
      f"(the recalibrated focal regimes are a separate group and are not among them): "
      f"{regime_names.SHORT_DISPLAY[_best.index[0]]} ({_best.iloc[0]:.4f}); "
      f"the baseline sits at {_u['auc_mean'].median():.4f}")

_r = all_summary[all_summary["regime"].isin(RECAL_REGIMES)]
_r_complete = _r[_r["recalibration_estimability_status"] == "complete"]
print(f"\ncross-fitted recalibration, {len(_r)} (regime, family, threshold) cells over the "
      f"four focal pipelines:")
print(f"  {int(RECAL_ESTIMATED.sum())} of {len(recalibrated_focal)} seed-level cells estimable")
print(f"  {len(_r_complete)} of {len(_r)} cells complete over all {len(SEEDS)} seeds")
if len(_r_complete):
    # Over COMPLETE cells only, and said so: a median that skipped blanked rows would be a
    # figure over whichever cells happened to be estimable, without saying which.
    print(f"  over those {len(_r_complete)}: median AUC "
          f"{_r_complete['auc_mean'].median():.4f}   "
          f"median ECE {_r_complete['ece_mean'].median():.4f}   "
          f"median calibration slope {_r_complete['cal_slope_mean'].median():.4f}")
if len(_r) > len(_r_complete):
    print(f"  {len(_r) - len(_r_complete)} incomplete cell(s) carry no across-seed value and "
          f"are excluded from the medians above")

print("\nNot computed, and therefore undetermined rather than settled:")
if not RUN_SOURCE_SCALED:
    print("  rung 1 of the baseline ladder — the contribution of cohort standardisation")

print("\nFitted their own models, and why:")
print("  Section J's curves       — one model per budget k, which is what the curve measures")
print("  Section K's sweep        — one update fit per ratio, on different folds of the anchor")
print("  Section M's batteries    — a different outcome, so a different model by definition")
print("  Section F's untuned arm  — a different configuration")

### What this run did not write

**Nothing else.** The screening frame, the selection review and coverage tables, the
configuration table, the gap frames, the budget curve, the calibration sweep, the structure-transfer summary and every figure
above are held in memory and displayed. None of them has a reader, and a file written for no
consumer invites the belief that something depends on it.

**The two battery files are restricted by content**, because they carry MCS-derived aggregate
rows at all three thresholds. Blanking the exact counts closes one recovery route; it is not
clearance. Every MCS-derived aggregate in them still requires disclosure review before it leaves
the approved environment, and no approval is implied by a directory, a filename or a passing
structural check.

**Loaded and not re-derived** — `spec/local_model_settings.csv`, and both cohorts' private
selection records under the working root when complete and current ones are there. A
Restart-and-Run-All re-derives everything else and does **not** re-run either consensus
search.

**A fresh search happens in exactly one case: no record for that cohort exists.** A record
that is incomplete, or made under a protocol identifier or preprocessing version that is no
longer live, is refused and left in place — nothing is topped up, silently replaced or partly
recomputed. Inspect it, then move or delete it deliberately to ask for a new complete search.
The tracked specification is never written by a run: `scripts/promote_local_settings.py` is its
only writer.

**Frozen and untouched** — everything under `outputs/`. Nothing here republishes over it. Four
of the frames above have a frozen published copy — the budget curve, the calibration sweep, the
structure-transfer summary and the model screening — and this run recomputes them without
overwriting anything.

### Open, for the notebook 03 handoff

Four artefacts notebook 03 or `src/tables.py` names have no producer in any notebook. Nothing
here creates a placeholder for them; whether they should exist is a question about notebook 03's
scientific purpose.

| artefact | who wants it |
|---|---|
| `yrbs_scores_by_budget.parquet` | `evaluation._score_sources` |
| `subgroup_scores.parquet` | `evaluation._score_sources` |
| `regime_significance.csv` | `tables.py`, as the source of `outputs/paper/data/transfer_significance.csv` |
| `screening_summary.csv` | `tables.py`, as the source of `outputs/paper/data/model_screening.csv` |

And two names to correct there: `loo_sensitivity_summary.csv` and `outcome_variants_summary.csv`
hold per-seed rows, so producer and consumer should be renamed together.

---
## The tables and figures this notebook offers the manuscript

Six tables, each built from a named list of columns. The exact counts the battery carries —
`n_test`, `n_pos` and the four confusion cells at each capacity — are not among them: they are
working quantities, and the MCS rows would be exact cell counts on restricted data.

The headline threshold leads; `>=1` and `>=3` travel in the same tables so the appendix can be
built from one file rather than three.

**PR-AUC always travels with its prevalence reference.** A PR-AUC without the prevalence it is
read against is not interpretable. The prevalence is a rate on the evaluation slice, and the
slice size is not published beside it, so the pair does not give a count back.

**Writing these is not clearance.** Every MCS-derived row needs review before it is committed.

In [ ]:
# The main publication tables report the headline threshold only.
PERFORMANCE_COLUMNS = [
    "regime",
    "family",
    "threshold",
    "n_seeds",
    "auc_mean",
    "auc_sd",
    "prauc_mean",
    "prauc_sd",
    "prevalence",
    "brier_mean",
    "brier_sd",
    "ece_mean",
    "ece_sd",
    "cal_slope_mean",
    "cal_intercept_mean",
]


main_performance = publication.require_primary_threshold(
    all_summary[
        (all_summary["arm"] == "tuned")
        & (all_summary["threshold"] == HEADLINE_THRESHOLD)
    ].copy(),
    HEADLINE_THRESHOLD,
    "main_model_performance",
)


# NO SELECTION-STATUS ACCOUNTING. The YRBS local reference takes one fixed configuration
# from the tracked specification, promoted after review and complete before this notebook
# begins, so every cell is configured on every split by construction. The aggregation,
# merge and partial-selection guard that stood here belonged to a per-split search this
# notebook no longer runs.


publication.save_table(
    main_performance,
    "main_model_performance.csv",
    PERFORMANCE_COLUMNS,
)

publication.save_table(
    main_performance,
    "transfer_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "auc_mean",
        "auc_sd",
        "transfer_loss",
    ],
)

# These quantities compare each procedure with the YRBS local reference. No
# cross-validated selection score appears: those are not held-out performance
# estimates, and the MCS ones are MCS-derived and stay in the private records.
publication.save_table(
    main_performance,
    "target_gap_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "auc_mean",
        "target_resource_gap",
        "adaptation_gain",
        "target_gap_recovered",
        "target_gap_reason",
    ],
)

publication.save_table(
    main_performance,
    "calibration_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "ece_mean",
        "ece_sd",
        "cal_slope_mean",
        "cal_intercept_mean",
        "brier_mean",
    ],
)

In [ ]:
# The label-budget curve is long-format: one row per (family, regime, threshold, k, metric).
label_budget = pd.concat([budget_unadapted, budget_target_only, budget_full_revision],
                         ignore_index=True)
publication.save_table(
    label_budget, "label_budget_results.csv",
    ["family", "regime", "threshold", "k", "metric", "mean", "sd", "n_seeds"])

publication.save_table(
    outcome_variants, "outcome_sensitivity.csv",
    [c for c in ("outcome", "model", "role", "auc", "seed") if c in outcome_variants.columns])

publication.save_table(
    loo_sensitivity, "leave_one_pillar_out.csv",
    [c for c in ("variant", "dropped_pillar", "model", "mcs_auc_internal",
                 "yrbs_auc_transfer", "transfer_gap", "seed")
     if c in loo_sensitivity.columns])

In [ ]:
# The three figures the manuscript uses from this notebook. Neutral filenames, no counts in
# any title or annotation. All three are MCS-derived through the source-reference line and
# remain subject to disclosure review.
publication.save_figure(transfer_figure, "transfer_comparison.png")
publication.save_figure(calibration_figure, "discrimination_calibration.png")
publication.save_figure(adaptation_figure, "focal_adaptation.png")

**Before committing anything from this notebook**, run
`python scripts/check_public_outputs.py`, then review every MCS-derived table and figure by
eye. The checker reports known patterns; it does not clear a result for release.

In [ ]:
primary_anchor_review = (
    all_summary.loc[
        all_summary["threshold"].eq(">=2")
        & all_summary["regime"].isin(
            ["mcs_internal", "unadapted", "yrbs_local"]
        ),
        [
            "family",
            "regime",
            "n_seeds",
            "auc_mean",
            "auc_sd",
            "prauc_mean",
            "prauc_sd",
            "brier_mean",
            "brier_sd",
            "ece_mean",
            "ece_sd",
            "cal_slope_mean",
            "cal_slope_sd",
        ],
    ]
    .sort_values(["family", "regime"])
    .reset_index(drop=True)
)

display(primary_anchor_review)